# HB（Heavy Ball 重球法）完全解析

HB（Heavy Ball 重球法），通常简称为动量梯度下降（Momentum Gradient Descent），是标准梯度下降法（GD）的第一次重大改进。其核心思想源自物理学中的**惯性**概念：通过累积历史梯度的指数加权平均来平滑更新方向，有效抑制震荡、加速收敛，并帮助优化器穿越鞍点和浅局部极小。

本教程聚焦于**HB（重球法）**，从直观动机出发，通过具体实例引出问题，系统讲解其数学原理、算法形式、收敛性理论与调参方法。关于 Nesterov 加速梯度（NAG），将在另一教程中单独讲解。

## 0. 术语约定

在优化理论中，"Momentum methods"（动量方法）是一个**大家族**，包含多个利用历史梯度信息来加速收敛的方法，其中最重要的成员包括**重球法（Heavy Ball, HB）**和 **Nesterov 加速梯度（NAG）**。

在深度学习社区（尤其是 PyTorch、TensorFlow 等框架的文档和代码中），"Momentum" 一词**默认特指重球法（HB）**。本教程遵循这一惯例：

| 术语 | 含义 |
| :--- | :--- |
| **重球法 / Heavy Ball / HB** | Polyak 于 1964 年提出的经典动量法，即狭义上的 Momentum |
| **动量法 / Momentum** | 在本教程中默认指代重球法（HB） |
| **Nesterov 加速梯度 / NAG** | 在另一教程中单独讲解 |

> **一句话总结**：本教程中所有 "Momentum" 或 "动量" 均指重球法（HB）。

## 1. 实例引出：GD的困境

### 1.0 目标函数的驻点分析

本实例目标函数为：

$$
f(x, y) = 0.2(x^2 - 1)^2 + y^2 - 0.15x
$$

其驻点及性质如下：

| 驻点坐标 | 类型 | 函数值 |
| :--- | :--- | :--- |
| $(-0.8882, 0)$ | 局部极小 | $0.156$ |
| $(-0.1949, 0)$ | 鞍点 | $0.094$ |
| $(1.0688, 0)$ | **全局极小** | $-0.316$ |

详细计算过程见**附录A：目标函数驻点分析**。

### 实验1

设迭代寻优起点为 $(-0.1949, 1.0000)$，则该点到目标函数三个驻点的距离(欧氏)为：

- 到局部极小值点 $(-0.8882, 0)$ 的距离$\approx 1.2168$
- 到鞍点 $(-0.1949, 0)$ 的距离$\approx 1.0000$
- 到全局极小值点 $(1.0831, 0)$ 的距离$\approx 1.6227$

起点到全局极小值点最远，到鞍点最近。


In [9]:
# 导入数值计算库 NumPy，用于处理数组、数学运算和生成网格数据
import numpy as np
# 导入 Plotly 的图形对象库，用于绘制交互式、高质量的图表
import plotly.graph_objects as go


# ============================================================
# 【全局超参配置区】
# ============================================================
# 定义一个字典，集中管理所有超参数，方便统一调整和修改
CONFIG = {
    # 迭代起始点 (公共参数，放在最前面)
    "start_x": -0.1949,        # 起始点的 x 坐标，设置在靠近鞍点 (-0.1949, 0) 的位置
    "start_y": 1.0,          # 起始点的 y 坐标，设置为较高的值，使起点处于高梯度区域
    
    # 算法参数
    "steps": 150,            # 两种算法运行的迭代步数
    "gd_lr": 0.07,           # 标准梯度下降 (GD) 的学习率 (alpha)
    "mom_lr": 0.13,          # Heavy Ball (HB) 法（动量法）的学习率 (alpha)
    "mom_beta": 0.94,        # HB 法的动量系数 (beta)，控制历史梯度对当前更新方向的影响程度
    
    # 绘图范围
    "x_min": -1.8,           # 绘图时 X 轴的最小值
    "x_max": 1.6,            # 绘图时 X 轴的最大值
    "y_min": -1.0,           # 绘图时 Y 轴的最小值 (扩大范围以防路径超出画布)
    "y_max": 1.3,            # 绘图时 Y 轴的最大值
    "mesh_res": 250,         # 绘制等高线/热力图时的网格分辨率，数值越大图像越平滑
    
    # 图片尺寸与样式
    "fig_height": 700,       # 图片的高度（像素）
    "fig_width": None,       # 图片的宽度，None 表示让 Plotly 自动适应
    "path_line_width": 1.2,  # 绘制优化路径线的宽度
    "path_marker_size": 4,   # 路径上散点标记的大小
    "title_x": 0.02,         # 图表标题的横向位置，0.02 表示左对齐并留有小边距
}

# 预先通过解析或数值方法求解得到的关键点坐标
local_min = (-0.8882, 0.0)     # 局部最优解坐标
saddle_pt = (-0.1949, 0.0)     # 鞍点坐标
global_min = (1.0831, 0.0)     # 全局最优解坐标


# ============================================================
# 第一部分：定义目标函数与梯度函数
# ============================================================

def f(x, y):
    """
    目标函数 f(x, y) = 0.2(x² - 1)² + y² - 0.15x
    这是一个非凸函数，包含局部最优、鞍点和全局最优。
    """
    # 返回该点处的函数值
    return 0.2 * (x**2 - 1)**2 + y**2 - 0.15 * x


def grad_f(x, y):
    """
    计算目标函数在点 (x, y) 处的梯度向量。
    梯度是函数在各方向上的偏导数，指向函数值增长最快的方向。
    """
    # 对 x 求偏导: ∂f/∂x = 0.8 * x * (x² - 1) - 0.15
    dx = 0.8 * x * (x**2 - 1) - 0.15
    # 对 y 求偏导: ∂f/∂y = 2 * y
    dy = 2 * y
    # 返回梯度向量 (dx, dy)
    return dx, dy


# ============================================================
# 第二部分：实现GD（标准梯度下降法）
# ============================================================

def vanilla_gd(start_x, start_y, lr, steps):
    """
    标准梯度下降法 (Gradient Descent)
    更新公式：x_{t+1} = x_t - lr * ∇f(x_t)
    """
    # 将传入的起点 x 坐标赋值给变量 x，作为当前迭代位置的 x 坐标
    x = start_x
    # 将传入的起点 y 坐标赋值给变量 y，作为当前迭代位置的 y 坐标
    y = start_y
    # 初始化路径列表，记录起始点的坐标和函数值
    path = [(x, y, f(x, y))]
    # 使用 for 循环迭代指定的步数，_ 表示不需要用到循环变量本身
    for _ in range(steps):
        # 调用 grad_f 函数计算当前点 (x, y) 的梯度
        gx, gy = grad_f(x, y)
        # 沿梯度的反方向更新 x 坐标：新位置 = 当前位置 - 学习率 * 梯度
        x = x - lr * gx
        # 沿梯度的反方向更新 y 坐标：新位置 = 当前位置 - 学习率 * 梯度
        y = y - lr * gy
        # 将更新后的位置 (x, y) 和函数值追加到路径列表中
        path.append((x, y, f(x, y)))
    # 将路径列表转换为 NumPy 数组并返回，方便后续切片处理
    return np.array(path)


# ============================================================
# 第三部分：实现HB法（Heavy Ball / 重球法 / 动量法）
# ============================================================

def hb_method(start_x, start_y, lr, beta, steps):
    """
    HB法（Heavy Ball Method），也称为动量梯度下降法。
    通过引入一个动量项（速度）来加速收敛并帮助跨越局部极小值或鞍点。
    更新公式：
        v_{t+1} = β * v_t + α * ∇f(x_t)   # 更新速度（动量）
        x_{t+1} = x_t - v_{t+1}            # 更新位置
    """
    # 将传入的起点 x 坐标赋值给变量 x，作为当前迭代位置的 x 坐标
    x = start_x
    # 将传入的起点 y 坐标赋值给变量 y，作为当前迭代位置的 y 坐标
    y = start_y
    # 初始化 x 方向的速度（动量）为 0，因为初始时没有历史梯度信息
    vx = 0.0
    # 初始化 y 方向的速度（动量）为 0，因为初始时没有历史梯度信息
    vy = 0.0
    # 初始化路径列表，记录起始点的坐标和函数值
    path = [(x, y, f(x, y))]
    # 使用 for 循环迭代指定的步数
    for _ in range(steps):
        # 调用 grad_f 函数计算当前点 (x, y) 的梯度
        gx, gy = grad_f(x, y)
        # 更新 x 方向的速度：新速度 = 动量系数 * 旧速度 + 学习率 * 当前梯度
        vx = beta * vx + lr * gx
        # 更新 y 方向的速度：新速度 = 动量系数 * 旧速度 + 学习率 * 当前梯度
        vy = beta * vy + lr * gy
        # 使用更新后的速度更新 x 坐标：当前位置 - 速度
        x = x - vx
        # 使用更新后的速度更新 y 坐标：当前位置 - 速度
        y = y - vy
        # 将更新后的位置 (x, y) 和函数值追加到路径列表
        path.append((x, y, f(x, y)))
    # 将路径列表转换为 NumPy 数组并返回
    return np.array(path)


# ============================================================
# 第四部分：运行优化算法
# ============================================================

# 从配置字典 CONFIG 中提取起点 x 坐标，赋值给变量 start_x
start_x = CONFIG["start_x"]
# 从配置字典 CONFIG 中提取起点 y 坐标，赋值给变量 start_y
start_y = CONFIG["start_y"]
# 从配置字典 CONFIG 中提取迭代步数，赋值给变量 common_steps
common_steps = CONFIG["steps"]

# 打印一行由 70 个等号组成的分隔线，用于美化输出
print("=" * 70)
# 打印标题【优化算法参数设置】
print("【优化算法参数设置】")
# 打印一行由 70 个等号组成的分隔线
print("=" * 70)
# 打印起点的坐标，使用 :.4f 格式保留 4 位小数
print(f"起点: ({start_x:.4f}, {start_y:.4f})")
# 打印起点处的函数值，使用 :.6f 格式保留 6 位小数
print(f"初始函数值: f({start_x:.4f}, {start_y:.4f}) = {f(start_x, start_y):.6f}")
# 从 CONFIG 中提取 GD 的学习率并打印
print(f"标准GD学习率: α = {CONFIG['gd_lr']}")
# 从 CONFIG 中提取 HB 法的学习率并打印
print(f"HB法学习率: α = {CONFIG['mom_lr']}")
# 从 CONFIG 中提取 HB 法的动量系数并打印
print(f"动量系数: β = {CONFIG['mom_beta']}")
# 打印迭代步数
print(f"迭代步数: {common_steps}")
# 打印局部最优点的坐标，f(*local_min) 将元组解包为两个参数传入 f 函数
print(f"局部最优: ({local_min[0]:.4f}, {local_min[1]:.4f}), f = {f(*local_min):.6f}")
# 打印鞍点的坐标和函数值
print(f"鞍点: ({saddle_pt[0]:.4f}, {saddle_pt[1]:.4f}), f = {f(*saddle_pt):.6f}")
# 打印全局最优点的坐标和函数值
print(f"全局最优: ({global_min[0]:.4f}, {global_min[1]:.4f}), f = {f(*global_min):.6f}")
# 打印一行分隔线
print("=" * 70)

# 调用 vanilla_gd 函数运行标准梯度下降法，传入起点x、起点y、GD学习率、迭代步数
path_gd = vanilla_gd(start_x, start_y, CONFIG["gd_lr"], common_steps)
# 调用 hb_method 函数运行 Heavy Ball 动量法，传入起点x、起点y、HB学习率、动量系数、迭代步数
path_mom = hb_method(start_x, start_y, CONFIG["mom_lr"], CONFIG["mom_beta"], common_steps)


# ============================================================
# 第五部分：生成等高线地形数据
# ============================================================

# 使用 np.linspace 在配置的 x 范围内生成均匀分布的坐标点
# 参数：起始值 CONFIG["x_min"]，结束值 CONFIG["x_max"]，点的数量 CONFIG["mesh_res"]
xs = np.linspace(CONFIG["x_min"], CONFIG["x_max"], CONFIG["mesh_res"])
# 使用 np.linspace 在配置的 y 范围内生成均匀分布的坐标点
ys = np.linspace(CONFIG["y_min"], CONFIG["y_max"], CONFIG["mesh_res"])

# 使用 np.meshgrid 将一维坐标数组 xs 和 ys 转换为二维网格坐标矩阵
# X, Y 是形状为 (mesh_res, mesh_res) 的二维数组，表示网格上的每个点
X, Y = np.meshgrid(xs, ys)

# 计算网格上每个点的函数值，得到 Z 矩阵（用于绘制等高线）
Z = f(X, Y)

# 从配置字典 CONFIG 中提取图片宽度，赋值给 FIG_WIDTH
FIG_WIDTH = CONFIG["fig_width"]
# 从配置字典 CONFIG 中提取图片高度，赋值给 FIG_HEIGHT
FIG_HEIGHT = CONFIG["fig_height"]
# 从配置字典 CONFIG 中提取路径线宽，赋值给 PATH_LW
PATH_LW = CONFIG["path_line_width"]
# 从配置字典 CONFIG 中提取路径标记大小，赋值给 PATH_MS
PATH_MS = CONFIG["path_marker_size"]
# 从配置字典 CONFIG 中提取标题横向位置，赋值给 TITLE_X
TITLE_X = CONFIG["title_x"]

# 计算鞍点处的函数值，存入变量 saddle_value
saddle_value = f(saddle_pt[0], saddle_pt[1])
# 计算起点处的函数值，存入变量 start_value
start_value = f(start_x, start_y)
# 计算局部最优点处的函数值，存入变量 local_min_value
local_min_value = f(local_min[0], local_min[1])
# 计算全局最优点处的函数值，存入变量 global_min_value
global_min_value = f(global_min[0], global_min[1])

# 打印起点函数值，保留 6 位小数
print(f"\n起点函数值: {start_value:.6f}")
# 打印鞍点函数值，保留 6 位小数
print(f"鞍点函数值: {saddle_value:.6f}")
# 打印局部最优值，保留 6 位小数
print(f"局部最优值: {local_min_value:.6f}")
# 打印全局最优值，保留 6 位小数
print(f"全局最优值: {global_min_value:.6f}")


# ============================================================
# 第六部分：构建等高线层级（等间距，必须包含鞍点）
# ============================================================

# 设置等高线的最小值为 -0.3
z_min = -0.3
# 设置等高线的最大值为 0.35
z_max = 0.35
# 设置等高线之间的步长间隔为 0.025
z_step = 0.025

# 将之前计算的鞍点函数值赋值给变量 saddle_level
saddle_level = saddle_value

# 计算从 z_min 到 saddle_level 需要多少个步长，结果可能不是整数
n = (saddle_level - z_min) / z_step
# 将 n 四舍五入到最近的整数，找到最接近的层级索引
n_rounded = round(n)
# 计算调整后的起始值，使得 saddle_level 恰好是某个层级
adjusted_start = saddle_level - n_rounded * z_step

# 使用调整后的起始值生成等间距的层级序列
contour_levels = np.arange(adjusted_start, z_max + z_step, z_step)
# 使用列表推导式过滤掉不在 [z_min, z_max] 范围内的层级
contour_levels = [level for level in contour_levels if z_min <= level <= z_max]

# 由于浮点数运算误差，鞍点值可能不在列表中，这里进行手动检查和插入
saddle_rounded = round(saddle_level, 6)
# 检查 contour_levels 中是否有某个值与鞍点值足够接近（误差小于1e-6）
if not any(abs(level - saddle_rounded) < 1e-6 for level in contour_levels):
    # 如果鞍点值不在列表中，则手动插入
    contour_levels.append(saddle_level)
    # 对列表进行排序，保持层级递增
    contour_levels = sorted(contour_levels)


# ============================================================
# 第七部分：稳定流形计算
# ============================================================

def find_stable_manifold():
    """
    计算稳定流形。
    稳定流形是垂直于鞍点脊线的方向，即沿 y 方向。
    在鞍点 (-0.1949, 0) 处，稳定流形是 x = -0.1949 的垂直线。
    
    对于函数 f(x,y) = 0.2(x²-1)² + y² - 0.15x
    Hessian矩阵：
    H = [[∂²f/∂x², ∂²f/∂x∂y],
         [∂²f/∂y∂x, ∂²f/∂y²]]
    
    在鞍点处：
    ∂²f/∂x² = 0.8(3x²-1) ≈ -0.709 (负 → 不稳定方向)
    ∂²f/∂y² = 2 (正 → 稳定方向)
    ∂²f/∂x∂y = 0
    
    所以稳定流形是沿 y 方向的垂直线：x = saddle_x
    """
    # 鞍点坐标
    saddle_x, saddle_y = saddle_pt
    
    # 生成稳定流形上的点（垂直线 x = saddle_x）
    y_vals = np.linspace(CONFIG["y_min"], CONFIG["y_max"], 200)
    stable_x = [saddle_x] * len(y_vals)
    stable_y = y_vals
    
    return stable_x, stable_y


# 计算稳定流形
stable_manifold_x, stable_manifold_y = find_stable_manifold()

# 打印稳定流形信息
print("\n" + "=" * 70)
print("【稳定流形】")
print("=" * 70)
print(f"鞍点坐标: ({saddle_pt[0]:.4f}, {saddle_pt[1]:.4f})")
print(f"稳定流形方程: x = {saddle_pt[0]:.4f} (垂直线)")
print(f"稳定流形方向: 沿y轴方向")
print(f"Hessian特征值: ∂²f/∂y² = 2 > 0 (正特征值, 稳定方向)")
print("=" * 70)


# ============================================================
# 第八部分：绘制图1 - GD法寻优路径（含稳定流形）
# ============================================================

# 创建一个 Plotly 图形对象，赋值给变量 fig1
fig1 = go.Figure()

# 添加带有颜色填充的等高线（Viridis色图，黑色统一线型，等间距，包含鞍点）
fig1.add_trace(go.Contour(
    x=xs,                        # 设置等高线的 x 轴坐标数据
    y=ys,                        # 设置等高线的 y 轴坐标数据
    z=Z,                         # 设置等高线的 z 轴数据（函数值矩阵）
    colorscale="Viridis",        # 设置颜色映射为 Viridis，用于背景色图填充
    contours=dict(
        start=contour_levels[0], # 等高线的起始值，必须包含鞍点所在的层级
        end=contour_levels[-1],  # 等高线的结束值
        size=z_step,             # 等高线的间距（等间距）
        showlabels=False,        # 不显示等高线上的数值标签，避免画面杂乱
        coloring='fill'          # 设置为 'fill'，表示用颜色填充等高线之间的区域，显示色图
    ),
    line=dict(width=1.2, color='black', dash='solid'),  # 统一线型：黑色实线，线宽 1.2
    showscale=True,              # 显示右侧的颜色条（Colorbar）
    opacity=0.7,                 # 设置透明度为 0.7，让路径线更清晰
    name="等间距等高线"           # 图例中显示的名称
))

# 添加稳定流形（灰色点线，不显眼）
fig1.add_trace(go.Scatter(
    x=stable_manifold_x,         # 稳定流形上的 x 坐标（固定值）
    y=stable_manifold_y,         # 稳定流形上的 y 坐标
    mode="lines",                # 仅显示线条
    line=dict(color="gray", width=2, dash="dot"),  # 灰色点线，线宽2
    name="稳定流形"              # 图例中显示的名称
))

# 添加 GD 算法的迭代路径（红色实线）
fig1.add_trace(go.Scatter(
    x=path_gd[:, 0],             # 取路径数组的第 0 列（所有点的 x 坐标）
    y=path_gd[:, 1],             # 取路径数组的第 1 列（所有点的 y 坐标）
    mode="lines+markers",        # 同时显示线条和标记点
    line=dict(color="red", width=PATH_LW),     # 红色实线，宽度从配置读取
    marker=dict(size=PATH_MS, color="red"),    # 红色标记点，大小从配置读取
    name="GD迭代路径"            # 图例中显示的名称
))

# 添加起点标记（黄色圆点）
fig1.add_trace(go.Scatter(
    x=[start_x],                 # 起点的 x 坐标
    y=[start_y],                 # 起点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=12, color="#ffcc00"),  # 大小为 12，颜色为黄色 (#ffcc00)
    name="起点"                  # 图例中显示的名称
))

# 添加局部极小值标记（橙色五角星）
fig1.add_trace(go.Scatter(
    x=[local_min[0]],            # 局部极小点的 x 坐标
    y=[local_min[1]],            # 局部极小点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=14, color="#ff8c00", symbol="star", line=dict(color="black", width=1)),
                                 # 大小为 14，颜色为橙色 (#ff8c00)，形状为五角星，黑色边框
    name="局部极小"           # 图例中显示的名称
))

# 添加鞍点标记（黑色加号）
fig1.add_trace(go.Scatter(
    x=[saddle_pt[0]],            # 鞍点的 x 坐标
    y=[saddle_pt[1]],            # 鞍点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=14, color="#000000", symbol="cross-thin", line=dict(color="black", width=1)),
                                 # 大小为 14，颜色为黑色 (#000000)，形状为加号 (cross)
    name="鞍点"               # 图例中显示的名称
))

# 添加全局极小值标记（绿色五角星）
fig1.add_trace(go.Scatter(
    x=[global_min[0]],           # 全局极小点的 x 坐标
    y=[global_min[1]],           # 全局极小点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=14, color="#00ff00", symbol="star", line=dict(color="black", width=1)),
                                 # 大小为 14，颜色为绿色 (#00ff00)，形状为五角星，黑色边框
    name="全局极小"           # 图例中显示的名称
))

# 更新图表的整体布局设置
fig1.update_layout(
    width=FIG_WIDTH,                         # 图片宽度
    height=FIG_HEIGHT,                       # 图片高度
    template="plotly_white",                 # 使用白色背景模板
    title=dict(text="图1：GD法寻优（含稳定流形）", x=TITLE_X, xanchor="left", font=dict(size=18)),
                                             # 标题文本、位置、对齐方式和字体大小
    xaxis_title="x",                         # X 轴标题
    yaxis_title="y",                         # Y 轴标题
    yaxis=dict(autorange=True),              # Y 轴自动缩放范围
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
                bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1, font=dict(size=11)),
                                             # 图例：水平排列、位于图表上方、半透明白色背景、黑色边框
    hovermode='closest',                     # 悬停时显示最近的数据点
    margin=dict(t=80, r=80)                  # 图表边距：顶部 80 像素，右侧 80 像素
)
# 显示图1
fig1.show()


# ============================================================
# 第九部分：绘制图2 - HB法寻优路径（含稳定流形）
# ============================================================

# 创建一个 Plotly 图形对象，赋值给变量 fig2
fig2 = go.Figure()

# 添加带有颜色填充的等高线（与图1完全一致）
fig2.add_trace(go.Contour(
    x=xs,                        # 设置等高线的 x 轴坐标数据
    y=ys,                        # 设置等高线的 y 轴坐标数据
    z=Z,                         # 设置等高线的 z 轴数据（函数值矩阵）
    colorscale="Viridis",        # 设置颜色映射为 Viridis，用于背景色图填充
    contours=dict(
        start=contour_levels[0], # 等高线的起始值，必须包含鞍点所在的层级
        end=contour_levels[-1],  # 等高线的结束值
        size=z_step,             # 等高线的间距（等间距）
        showlabels=False,        # 不显示等高线上的数值标签，避免画面杂乱
        coloring='fill'          # 设置为 'fill'，表示用颜色填充等高线之间的区域，显示色图
    ),
    line=dict(width=1.2, color='black', dash='solid'),  # 统一线型：黑色实线，线宽 1.2
    showscale=True,              # 显示右侧的颜色条（Colorbar）
    opacity=0.7,                 # 设置透明度为 0.7，让路径线更清晰
    name="等间距等高线"           # 图例中显示的名称
))

# 添加稳定流形（灰色点线，不显眼）
fig2.add_trace(go.Scatter(
    x=stable_manifold_x,         # 稳定流形上的 x 坐标（固定值）
    y=stable_manifold_y,         # 稳定流形上的 y 坐标
    mode="lines",                # 仅显示线条
    line=dict(color="gray", width=2, dash="dot"),  # 灰色点线，线宽2
    name="稳定流形"              # 图例中显示的名称
))

# 添加 HB 算法的迭代路径（红色实线）
fig2.add_trace(go.Scatter(
    x=path_mom[:, 0],            # 取路径数组的第 0 列（所有点的 x 坐标）
    y=path_mom[:, 1],            # 取路径数组的第 1 列（所有点的 y 坐标）
    mode="lines+markers",        # 同时显示线条和标记点
    line=dict(color="red", width=PATH_LW),     # 红色实线，宽度从配置读取
    marker=dict(size=PATH_MS, color="red"),    # 红色标记点，大小从配置读取
    name="HB迭代路径"            # 图例中显示的名称
))

# 添加起点标记（黄色圆点）
fig2.add_trace(go.Scatter(
    x=[start_x],                 # 起点的 x 坐标
    y=[start_y],                 # 起点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=12, color="#ffcc00"),  # 大小为 12，颜色为黄色 (#ffcc00)
    name="起点"                  # 图例中显示的名称
))

# 添加局部极小值标记（橙色五角星）
fig2.add_trace(go.Scatter(
    x=[local_min[0]],            # 局部极小点的 x 坐标
    y=[local_min[1]],            # 局部极小点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=14, color="#ff8c00", symbol="star", line=dict(color="black", width=1)),
                                 # 大小为 14，颜色为橙色 (#ff8c00)，形状为五角星，黑色边框
    name="局部极小"           # 图例中显示的名称
))

# 添加鞍点标记（黑色加号）
fig2.add_trace(go.Scatter(
    x=[saddle_pt[0]],            # 鞍点的 x 坐标
    y=[saddle_pt[1]],            # 鞍点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=14, color="#000000", symbol="cross-thin", line=dict(color="black", width=1)),
                                 # 大小为 14，颜色为黑色 (#000000)，形状为加号 (cross)
    name="鞍点"               # 图例中显示的名称
))

# 添加全局极小值标记（绿色五角星）
fig2.add_trace(go.Scatter(
    x=[global_min[0]],           # 全局极小点的 x 坐标
    y=[global_min[1]],           # 全局极小点的 y 坐标
    mode="markers",              # 仅显示标记
    marker=dict(size=14, color="#00ff00", symbol="star", line=dict(color="black", width=1)),
                                 # 大小为 14，颜色为绿色 (#00ff00)，形状为五角星，黑色边框
    name="全局极小"           # 图例中显示的名称
))

# 更新图表的整体布局设置
fig2.update_layout(
    width=FIG_WIDTH,                         # 图片宽度
    height=FIG_HEIGHT,                       # 图片高度
    template="plotly_white",                 # 使用白色背景模板
    title=dict(text="图2：HB法寻优（含稳定流形）", x=TITLE_X, xanchor="left", font=dict(size=18)),
                                             # 标题文本、位置、对齐方式和字体大小
    xaxis_title="x",                         # X 轴标题
    yaxis_title="y",                         # Y 轴标题
    yaxis=dict(autorange=True),              # Y 轴自动缩放范围
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
                bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1, font=dict(size=11)),
                                             # 图例：水平排列、位于图表上方、半透明白色背景、黑色边框
    hovermode='closest',                     # 悬停时显示最近的数据点
    margin=dict(t=80, r=80)                  # 图表边距：顶部 80 像素，右侧 80 像素
)
# 显示图2
fig2.show()


# ============================================================
# 第十部分：打印等高线层级信息
# ============================================================

# 打印一个空行，用于分隔输出
print("\n" + "=" * 70)
# 打印标题 "等高线层级（等间距，包含鞍点）:"
print("等高线层级（等间距，包含鞍点）:")
# 打印一行分隔线
print("=" * 70)
# 遍历 contour_levels 列表中的每一个层级值
for level in contour_levels:
    # 打印每个层级值，保留 6 位小数，前面加两个空格缩进
    print(f"  {level:.6f}")
# 打印一行分隔线
print("=" * 70)


# ============================================================
# 第十一部分：结果分析
# ============================================================
print("\n" + "=" * 70)
print("【算法结果分析】")
print("=" * 70)
print(f"GD 法最终位置: ({path_gd[-1, 0]:.4f}, {path_gd[-1, 1]:.4f})")
print(f"GD 法最终函数值: {path_gd[-1, 2]:.6f}")
print(f"HB 法最终位置: ({path_mom[-1, 0]:.4f}, {path_mom[-1, 1]:.4f})")
print(f"HB 法最终函数值: {path_mom[-1, 2]:.6f}")
print()
print("稳定流形分析:")
print("  - 稳定流形是沿 y 方向的垂直线，方程为 x = -0.1949")
print("  - 在鞍点处，沿 y 方向是极小值（Hessian 正特征值）")
print("  - 如果优化算法从稳定流形上的点出发，会沿着 y 方向远离鞍点")
print("  - 稳定流形将空间分为左、右两个区域，但算法不易穿越该线")
print("=" * 70)

【优化算法参数设置】
起点: (-0.1949, 1.0000)
初始函数值: f(-0.1949, 1.0000) = 1.214329
标准GD学习率: α = 0.07
HB法学习率: α = 0.13
动量系数: β = 0.94
迭代步数: 150
局部最优: (-0.8882, 0.0000), f = 0.142143
鞍点: (-0.1949, 0.0000), f = 0.214329
全局最优: (1.0831, 0.0000), f = -0.156472

起点函数值: 1.214329
鞍点函数值: 0.214329
局部最优值: 0.142143
全局最优值: -0.156472

【稳定流形】
鞍点坐标: (-0.1949, 0.0000)
稳定流形方程: x = -0.1949 (垂直线)
稳定流形方向: 沿y轴方向
Hessian特征值: ∂²f/∂y² = 2 > 0 (正特征值, 稳定方向)



等高线层级（等间距，包含鞍点）:
  -0.285671
  -0.260671
  -0.235671
  -0.210671
  -0.185671
  -0.160671
  -0.135671
  -0.110671
  -0.085671
  -0.060671
  -0.035671
  -0.010671
  0.014329
  0.039329
  0.064329
  0.089329
  0.114329
  0.139329
  0.164329
  0.189329
  0.214329
  0.239329
  0.264329
  0.289329
  0.314329
  0.339329

【算法结果分析】
GD 法最终位置: (-0.1893, 0.0000)
GD 法最终函数值: 0.214318
HB 法最终位置: (1.0920, -0.0098)
HB 法最终函数值: -0.156295

稳定流形分析:
  - 稳定流形是沿 y 方向的垂直线，方程为 x = -0.1949
  - 在鞍点处，沿 y 方向是极小值（Hessian 正特征值）
  - 如果优化算法从稳定流形上的点出发，会沿着 y 方向远离鞍点
  - 稳定流形将空间分为左、右两个区域，但算法不易穿越该线


In [ ]:

# ============================================================
# 第八部分：绘制图3 - 迭代收敛曲线
# ============================================================

# 提取算法运行过程中的损失值 (即数组的第2列，函数值)
loss_gd = path_gd[:, 2]
loss_mom = path_mom[:, 2]

# 生成步数序列，用于作为 X 轴坐标 (0, 1, 2, ...)
steps_arr = np.arange(len(loss_gd))

# 计算全局最优点的函数值，用于画虚线参考
optimal_loss = f(*global_min)

fig3 = go.Figure()  # 创建图形3

# 绘制 GD 的损失曲线
fig3.add_trace(go.Scatter(
    x=steps_arr, y=loss_gd,
    mode="lines+markers",
    line=dict(color="#1f77b4", width=1.0),   # 蓝色线
    marker=dict(size=2, color="#1f77b4"),
    name="GD法"
))

# 绘制 HB 的损失曲线
fig3.add_trace(go.Scatter(
    x=steps_arr, y=loss_mom,
    mode="lines+markers",
    line=dict(color="#d62728", width=1.0),   # 红色线
    marker=dict(size=2, color="#d62728"),
    name="HB法"
))

# 绘制全局最优损失值的水平虚线
fig3.add_trace(go.Scatter(
    x=[0, steps_arr[-1]],                    # X 轴从起点到终点
    y=[optimal_loss, optimal_loss],          # Y 轴恒定为最优损失
    mode="lines",
    line=dict(color="green", width=1.0, dash="dash"),  # 绿色虚线
    name=f"全局最优损失 ({optimal_loss:.4f})"
))

# 在图上添加 GD 终点值的文字注释
fig3.add_annotation(
    x=steps_arr[-1] * 0.95,                  # 放在靠近横轴终点的位置
    y=loss_gd[-1] + 0.02,                    # 稍微向上偏移，避免重叠
    text=f"GD法: {loss_gd[-1]:.4f}",
    showarrow=False,                         # 不显示箭头
    font=dict(color="#1f77b4", size=12)
)

# 在图上添加 HB 终点值的文字注释
fig3.add_annotation(
    x=steps_arr[-1] * 0.95,
    y=loss_mom[-1] - 0.03,                   # 稍微向下偏移
    text=f"HB法: {loss_mom[-1]:.4f}",
    showarrow=False,
    font=dict(color="#d62728", size=12)
)

# 更新图3布局
fig3.update_layout(
    width=FIG_WIDTH, height=FIG_HEIGHT,
    template="plotly_white",
    title=dict(text="图3：GD法 vs HB法：迭代收敛曲线对比", x=TITLE_X, xanchor="left", font=dict(size=16)), # 使用统一的标题横向位置
    xaxis_title="迭代步数", yaxis_title="损失值 f(x,y)",
    yaxis=dict(autorange=True),              # 关键：删除固定 range，开启自动缩放
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02,
        xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.9)",     # 图例背景半透明
        bordercolor="lightgray", borderwidth=1
    )
)
fig3.show()  # 显示图3

### 1.1 实例结果解读：逃逸能力的直观展示

从可视化结果可以清晰看到：

1. **GD法**从起点 `(-0.1949, 1.0)` 出发，沿着局部梯度方向滑入最近的"坑"——**鞍点** `(-0.1949, 0)`，之后梯度趋近于零，更新几乎停滞，永远无法到达右侧更深的**全局最优** `(1.0831, 0)`。

2. **HB法**从同一起点出发，凭借累积的速度惯性：
   - 冲过了鞍点区域；
   - 穿越了梯度接近零的稳定流形；
   - 最终抵达全局最优，损失显著更低。

这正是重球法的核心价值之一：**用历史梯度信息赋予优化器"惯性"，使其能够摆脱局部陷阱，加速向全局最优迈进。**

## 2. GD的三大痛点

GD在非凸优化中存在三个核心问题：

| 问题 | 表现 | 后果 |
| :--- | :--- | :--- |
| **锯齿振荡** | 在狭窄的峡谷区域，梯度方向在谷壁间来回振荡 | 前进缓慢，效率低下 |
| **平坦区域停滞** | 在梯度极小的平坦区域，参数几乎不动 | 训练陷入停滞 |
| **局部极小值陷阱** | 容易困在较差的局部极小值中 | 无法到达全局最优 |

重球法的核心思想正是针对这三大痛点设计的：通过引入惯性（速度的持续积累），**抑制锯齿振荡**（方向平滑）、**穿越平坦区域**（速度惯性维持）、**冲出局部极小**（动能冲过浅坑）。

## 3. 物理动机：从"局部最速"到"惯性修正"

### 3.1 GD法只是"局部最速"，不是"全局最速"

GD法的更新公式：

$$
x_{t+1} = x_t - \alpha \nabla f(x_t)
$$

它每一步选择的方向，是**当前点** $x_t$ 处方向导数最小的方向。这个"最快"只对**无穷小步长**严格成立。一旦迈出有限的一步，地形已经变了，原先的"最速方向"就不再是通往全局最优的最优路径。

用一句话概括：**GD法是"近视"的——它只看得见脚下，看不见远方。**

### 3.2 一个形象的类比：闭眼走山路

想象你站在一座崎岖的山丘上，目标是走到最低的山谷。GD法的策略是：

1. 闭上眼睛，只感受**脚下**的坡度；
2. 朝最陡的方向迈出一步；
3. 睁开眼，重新感受坡度；
4. 重复以上过程。

这个策略在**光滑的碗状地形**上效果很好。但在**非凸地形**上：

- 如果脚下是一个**浅坑**，你会顺着坡度走进去，然后发现四周都是上坡，再也走不出来；
- 如果遇到一片**平坦的高原**，坡度几乎为零，你会不知道该往哪走，只能原地打转；
- 如果走在**狭窄的峡谷**里，坡度在两壁之间来回反弹，你会像乒乓球一样左右震荡，前进极慢。

GD法之所以会这样，正是因为它**每一步都重新做一次"局部最速"决策**，没有任何对历史方向的记忆。

### 3.3 重球法的核心思想：给"局部最速"加上"惯性"

重球法的出发点很简单：既然GD法的"近视"源于它**只依赖当前梯度**，那就让它**同时记住过去的方向**。

于是我们引入一个"速度"变量 $v$，把更新拆成两步。这就是**重球法（Heavy Ball, HB）**的标准形式：

$$
\boxed{\begin{aligned}
v_{t+1} &= \beta v_t - \alpha \nabla f(x_t) \\
x_{t+1} &= x_t + v_{t+1}
\end{aligned}}
$$

**直觉理解**：

- $v_t$ 是"当前的运动趋势"——它记得过去一直在往哪个方向走
- $\beta$ 是"惯性系数"——每走一步，旧趋势保留多少
- $-\alpha \nabla f(x_t)$ 是"当前的修正力"——地形告诉我们应该往哪拐

新方向 = **惯性方向** 与 **局部最速方向** 的加权折中。

关键在于：**重球法不再追求每一步都是"局部最速"**。它愿意在当前这一步稍微偏离负梯度方向，以换取整条路径的更快收敛。这是一种用"单步次优"换"全局更优"的策略。

### 3.4 为什么"非最速"反而更快？

答案是：**"局部最速"不等于"全局最优路径"**。

**场景1：峡谷震荡**

在狭窄的峡谷中，负梯度方向在两侧谷壁之间来回反弹。重球法通过累积历史方向，让**横向震荡相互抵消**，**纵向进展持续积累**——路径被"平滑"了。

**场景2：平坦高原**

在梯度接近零的平坦区域，GD法几乎原地不动。重球法虽然每一步的梯度修正很小，但**惯性项 $\beta v_t$ 仍然保留着之前积累的速度**，可以推动参数继续前进。

**场景3：浅坑陷阱**

当GD法滑入一个浅的局部极小值时，梯度趋近于零，更新停滞。重球法则带着之前积累的"动能"，即使梯度已经很小，速度也不会立即消失——它可能**冲过浅坑**，继续向更深的全局最优前进。

在这三种场景中，重球法每一步的瞬时下降率可能**不如**GD法，但整条路径的累积下降量**远大于**GD法。这正是"局部次优换全局更优"的精髓。

## 4. HB法的寻优性能：系统全面解析

**HB（Heavy Ball Method, 重球法）**，是 Polyak 于 1964 年提出的一阶优化算法。其核心思想源自经典力学中"重球在阻尼介质中滚动"的物理模型：一个具有质量的球在重力场中滚动时，由于惯性，它不会在每一个瞬时都严格沿最陡下降方向运动，而是倾向于保持原有的运动方向，从而在峡谷地形中比"轻球"（无惯性）滚得更快、更稳。

对于目标函数 $f: \mathbb{R}^n \to \mathbb{R}$，HB法的标准形式为：

$$
\boxed{\begin{aligned}
v_{t+1} &= \beta v_t - \alpha \nabla f(x_t) \\
x_{t+1} &= x_t + v_{t+1}
\end{aligned}}
$$

其中：

| 符号 | 名称 | 含义 |
| :--- | :--- | :--- |
| $x_t \in \mathbb{R}^n$ | 参数向量 | 第 $t$ 步迭代点 |
| $v_t \in \mathbb{R}^n$ | 速度向量（动量项） | 历史梯度的指数滑动平均，初始 $v_0 = \mathbf{0}$ |
| $\alpha > 0$ | 学习率 | 当前梯度对速度的贡献强度 |
| $\beta \in [0, 1)$ | 动量系数（衰减率） | 上一时刻速度的保留比例，控制惯性强度 |
| $\nabla f(x_t)$ | 梯度 | 当前点处的一阶导数值 |

> **符号约定**：在深度学习框架（如 PyTorch）中，通常将负号吸收进速度定义，写作等价形式：
>
> $$
> v_{t+1} = \beta v_t + \nabla f(x_t), \quad x_{t+1} = x_t - \alpha v_{t+1}
> $$
>
> 两种形式数学等价，本文统一采用框图中的物理符号形式。

### 4.1 寻优性能全景概览

HB法的寻优性能可以从**收敛速度、路径平滑性、逃逸能力和鲁棒性**四个维度系统理解。下表给出与GD法的全景对比：

| 性能维度 | GD法 | HB法 |
| :--- | :--- | :--- |
| **收敛速度（强凸二次函数）** | $O(\kappa \log(1/\epsilon))$ | $O(\sqrt{\kappa} \log(1/\epsilon))$ ✅ **平方根加速** |
| **病态条件（$\kappa$大）** | 严重受制约 | **平方根加速** ✅ |
| **锯齿振荡抑制** | 差（剧烈振荡） | **好**（横向抵消、纵向积累） ✅ |
| **鞍点逃逸** | 弱（梯度趋零停滞） | **较好**（惯性推动前进） ✅ |
| **浅局部极小逃逸** | 弱（容易被困） | **好**（动能冲过） ✅ |
| **步长安全区间** | $0 < \alpha < 2/L$ | $0 < \alpha < 2(1+\beta)/L$（**更宽**） ✅ |
| **学习率鲁棒性** | 较差 | **更好**（安全区间宽近一倍） ✅ |
| **过冲/超调** | 无 | 有（惯性代价） |
| **额外计算/存储** | 无 | 一份速度向量 |

> **说明**：上表中"平方根加速"的理论保证**仅对强凸二次函数严格成立**；振荡抑制、鞍点逃逸、学习率鲁棒性等优势在更一般的函数类上主要体现为经验性效果。理论保证的精确边界见**附录I**。

### 4.2 核心优势一：沿低曲率方向加速收敛

在深度学习的损失函数中，不同参数方向上的曲率差异极大（**病态条件问题**，详见附录A）。GD法在曲率小的方向（平缓方向）步进很小，导致收敛极慢。HB法通过累积历史梯度，使得**平缓方向上的有效步长持续增大**，显著加速收敛。

**定量分析**：对于**强凸二次函数** $f(x) = \frac{1}{2}x^T Q x$，条件数 $\kappa = \lambda_{\max}/\lambda_{\min}$：

| 算法 | 收敛速率 | 条件数很大时的表现 |
| :--- | :--- | :--- |
| GD法 | $O(\kappa \log(1/\epsilon))$ | 条件数每增大10倍，所需迭代步数约增大10倍 |
| HB法（最优动量） | $O(\sqrt{\kappa} \log(1/\epsilon))$ | 条件数每增大10倍，所需迭代步数仅约增大 $\sqrt{10} \approx 3.16$ 倍 |

**有效步长机制**：动量累积的稳态等效有效步长约为 $\alpha/(1-\beta)$。当 $\beta=0.9$ 时，有效步长约为学习率的10倍——这解释了其加速能力。

**直观理解**：如同在平坦的高速公路上逐渐加速的汽车——虽然瞬时"油门"（梯度）很小，但速度在持续累积。

> **理论边界提示**：上述平方根加速的严格理论保证**仅对强凸二次函数成立**。4.4 节的实验正是这一情形的直接验证。对一般强凸光滑函数，HB 没有同等的全局加速保证，详见附录I。

### 4.3 核心优势二：高曲率方向抑制振荡

在曲率大的方向（陡峭方向），梯度剧烈变化导致GD法产生锯齿状振荡。动量项起到了**低通滤波/阻尼作用**——它平均了反向的梯度信号，削弱了高频振荡分量，使下降轨迹更平滑。

**信号处理视角**：动量更新 $v_{t+1} = \beta v_t + \alpha \nabla f(x_t)$ 本质上是一个**一阶低通滤波器**：

- 输入：梯度序列 $\{\nabla f(x_t)\}$
- 输出：速度序列 $\{v_t\}$（用于参数更新）
- 频率响应：低频分量（持续趋势）保留，高频分量（剧烈振荡）被衰减

**峡谷场景分析**：在狭窄峡谷中，横向梯度符号交替（$+g, -g, +g, -g, ...$），动量累积使它们相互抵消；纵向梯度方向一致（$-g, -g, -g, ...$），动量累积使它们持续叠加。结果是横向震荡被抑制，纵向进展被加速。

### 4.4 实验：验证上述两大核心优势

下面通过一个精心设计的二维病态二次函数实验（关于"病态"的详细解释见附录B），**同时定量验证**4.2节和4.3节的两个核心优势：

- **x方向（低曲率/平缓）**：验证4.2节"沿低曲率方向加速收敛"
- **y方向（高曲率/陡峭）**：验证4.3节"高曲率方向抑制振荡"

这个实验的巧妙之处在于：函数的两个方向**完全解耦**，使得我们可以独立观察HB法在每个方向上的行为机制。

> **本节实验对应 4.1 节全景概览中的哪几条？**
>
> 本实验的目标函数 $f(x,y)=\frac{1}{2}(x^2+100y^2)$ 是**强凸二次函数**（$A = \text{diag}(1,100) \succ 0$），因此**满足平方根加速的理论保证条件**。它直接验证了 4.1 节中的：
>
> | 4.1 节条目 | 本实验中的体现 |
> | :--- | :--- |
> | **收敛速度（强凸二次函数）** ✅ | 相同步数下 HB 法在 x 方向前进距离远大于 GD 法 |
> | **病态条件（$\kappa$大）** ✅ | $\kappa=100$，HB 法有效步长放大 $1/(1-\beta)=10$ 倍 |
> | **锯齿振荡抑制** ✅ | y 方向（陡峭）梯度符号交替时，HB 法速度相互抵消，保持稳定 |
> | **步长安全区间** ✅ | $\alpha=0.018 < 2(1+\beta)/\lambda_{\max} = 0.038$，HB 法稳定 |

In [16]:
"""
二维病态二次函数：HB法沿低曲率方向加速收敛演示
==================================================
目标函数：f(x, y) = 0.5 * (x^2 + 100 * y^2)
- 条件数 κ = λmax/λmin = 100/1 = 100
  · x方向的二阶导数为1（曲率小，地形平缓，等高线稀疏）
  · y方向的二阶导数为100（曲率大，地形陡峭，等高线密集）
- 全局最优：(0, 0)，f(0,0) = 0

实验目的：
对比GD法与HB法在【相同学习率】、【相同迭代步数】下的收敛效果。
同时验证HB法的两大核心优势：
  1. 沿低曲率（平缓）方向加速收敛
  2. 高曲率（陡峭）方向抑制振荡
"""

# 导入NumPy库用于数值计算
import numpy as np
# 导入Plotly图形库用于可视化
import plotly.graph_objects as go

# ============================================================
# 第一部分：定义目标函数与梯度
# ============================================================

def f_ill(x, y):
    """
    病态二次函数 f(x, y) = 0.5 * (x² + 100·y²)
    
    Hessian矩阵：H = [[1, 0], [0, 100]]
    特征值：λmin = 1（x方向），λmax = 100（y方向）
    条件数：κ = 100（高度病态）
    
    参数：
        x, y: 二维坐标点
    返回：
        函数值
    """
    # 计算二次函数值：x方向曲率为1，y方向曲率为100
    return 0.5 * (x**2 + 100 * y**2)


def grad_f_ill(x, y):
    """
    梯度：∂f/∂x = x, ∂f/∂y = 100·y
    
    参数：
        x, y: 二维坐标点
    返回：
        (∂f/∂x, ∂f/∂y)
    """
    # x方向梯度为x，y方向梯度为100*y
    return x, 100 * y


# ============================================================
# 第二部分：实现GD法与HB法
# ============================================================

def vanilla_gd_ill(x0, y0, lr, steps=60):
    """
    GD法（标准梯度下降法）
    
    更新公式：x_{t+1} = x_t - lr * gradient
    
    参数：
        x0, y0: 起始坐标
        lr: 学习率
        steps: 迭代步数
    返回：
        numpy数组，包含每步的坐标路径
    """
    # 初始化当前位置
    x, y = x0, y0
    # 初始化路径列表
    path = [(x, y)]
    # 执行steps次迭代
    for t in range(steps):
        # 计算梯度
        gx, gy = grad_f_ill(x, y)
        # 沿负梯度方向更新
        x = x - lr * gx
        y = y - lr * gy
        # 记录新位置
        path.append((x, y))
    # 转换为numpy数组并返回
    return np.array(path)


def HB_ill(x0, y0, lr, beta, steps=60):
    """
    HB法（Heavy Ball Method / 重球法）
    
    更新公式：
        v_new = beta * v + lr * grad
        x_new = x - v_new
    
    参数：
        x0, y0: 起始坐标
        lr: 学习率
        beta: 动量系数
        steps: 迭代步数
    返回：
        numpy数组，包含每步的坐标路径
    """
    # 初始化当前位置
    x, y = x0, y0
    # 初始化速度向量为0
    vx, vy = 0.0, 0.0
    # 初始化路径列表
    path = [(x, y)]
    # 执行steps次迭代
    for t in range(steps):
        # 计算梯度
        gx, gy = grad_f_ill(x, y)
        # 更新速度：保留历史速度的beta比例，加上当前梯度的lr倍
        vx = beta * vx + lr * gx
        vy = beta * vy + lr * gy
        # 更新位置：沿速度方向前进
        x = x - vx
        y = y - vy
        # 记录新位置
        path.append((x, y))
    # 转换为numpy数组并返回
    return np.array(path)


# ============================================================
# 第三部分：运行实验（相同学习率、相同步数）
# ============================================================

# 设置起点：x远离最优（-8），y接近最优（0.1）
# 这样设置是为了突出x方向（平缓方向）的收敛瓶颈
start_ill = (-8.0, 0.1)

# 学习率：略小于y方向稳定上限 2/λmax = 2/100 = 0.02
# 取0.018保证GD法在y方向不会发散
lr_ill = 0.018

# 动量系数：经典默认值0.9
beta_ill = 0.9

# 迭代步数：15步，足够看出差异
steps_ill = 15

# 运行GD法
path_gd_ill = vanilla_gd_ill(*start_ill, lr=lr_ill, steps=steps_ill)
# 运行HB法
path_mom_ill = HB_ill(*start_ill, lr=lr_ill, beta=beta_ill, steps=steps_ill)


# ============================================================
# 第四部分：生成等高线地形数据
# ============================================================

# 在x轴上生成网格点：从-10到10，共600个点
xs_ill = np.linspace(-10, 10, 600)
# 在y轴上生成网格点：从-1.5到1.5，共600个点
ys_ill = np.linspace(-1.5, 1.5, 600)
# 生成二维网格坐标矩阵
X_ill, Y_ill = np.meshgrid(xs_ill, ys_ill)
# 计算网格上每个点的函数值
Z_ill = f_ill(X_ill, Y_ill)


# ============================================================
# 第五部分：绘制等高线填充图 + 优化路径
# ============================================================

# 创建Plotly图形对象
fig_ill = go.Figure()

# 添加等高线填充图
fig_ill.add_trace(go.Contour(
    x=xs_ill, y=ys_ill, z=Z_ill,  # 网格数据
    colorscale="Viridis",  # 颜色方案
    contours=dict(
        coloring="fill",  # 填充模式
        showlabels=True,  # 显示等高线标签
        labelfont=dict(size=10, color="white"),  # 标签字体
        start=0.1, end=60, size=3  # 等高线范围从0.1到60，间隔为3
    ),
    line=dict(width=0.8, color="white"),  # 等高线边框为白色
    showscale=True,  # 显示颜色条
    colorbar=dict(title="f(x,y)", thickness=20),  # 颜色条标题和厚度
    opacity=0.85,  # 透明度
    name="损失地形"
))

# 添加GD法的迭代路径
fig_ill.add_trace(go.Scatter(
    x=path_gd_ill[:, 0], y=path_gd_ill[:, 1],  # 路径坐标
    mode="lines+markers",  # 线条加标记
    line=dict(color="red", width=1),  # 红色线条
    marker=dict(size=2, color="red"),  # 红色小标记
    name=f"GD法 (lr={lr_ill}, {steps_ill}步)"
))

# 添加HB法的迭代路径
fig_ill.add_trace(go.Scatter(
    x=path_mom_ill[:, 0], y=path_mom_ill[:, 1],
    mode="lines+markers",
    line=dict(color="cyan", width=1),  # 青色线条
    marker=dict(size=2, color="cyan"),  # 青色标记
    name=f"HB法 (lr={lr_ill}, β={beta_ill}, {steps_ill}步)"
))

# 标记起点位置（红色五角星）
fig_ill.add_trace(go.Scatter(
    x=[start_ill[0]], y=[start_ill[1]],
    mode="markers",
    marker=dict(size=8, color="red", symbol="star"),
    name=f"起点 ({start_ill[0]}, {start_ill[1]})"
))

# 标记全局最优位置（绿色菱形）
fig_ill.add_trace(go.Scatter(
    x=[0], y=[0],
    mode="markers",
    marker=dict(size=6, color="lime", symbol="diamond"),
    name="全局最优 (0, 0)"
))

# 设置图表布局
fig_ill.update_layout(
    width=1000, height=750,  # 图表尺寸
    template="plotly_white",
    title=dict(
        text=f"病态二次函数（κ=100）：相同{steps_ill}步下，HB法收敛更快",
        x=0.5, xanchor="center", font=dict(size=18)
    ),
    xaxis_title="x（低曲率/平缓方向）",
    yaxis_title="y（高曲率/陡峭方向）",
    xaxis=dict(range=[-10, 10]),  # x轴范围
    yaxis=dict(range=[-1.5, 1.5]),  # y轴范围
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02,  # 图例水平居中于顶部
        xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.95)",  # 半透明白色背景
        bordercolor="lightgray", borderwidth=1,
        font=dict(size=13)
    )
)
# 显示图表
fig_ill.show()


# ============================================================
# 第六部分：打印收敛结果
# ============================================================

# 打印详细的收敛结果对比
print("=" * 70)
print("【病态二次函数 κ=100：GD法 vs HB法】")
print(f"相同学习率 α={lr_ill}，相同步数 {steps_ill}")
print("=" * 70)
print(f"起点: ({start_ill[0]}, {start_ill[1]}), f={f_ill(*start_ill):.4f}")
print("-" * 70)

# 打印GD法的终点结果
print(f"GD法  终点: x={path_gd_ill[-1,0]:.8f}, y={path_gd_ill[-1,1]:.8f}")
print(f"         f = {f_ill(*path_gd_ill[-1]):.10f}")
print(f"         距最优距离 = {np.sqrt(path_gd_ill[-1,0]**2 + path_gd_ill[-1,1]**2):.6f}")
print("-" * 70)

# 打印HB法的终点结果
print(f"HB法  终点: x={path_mom_ill[-1,0]:.8f}, y={path_mom_ill[-1,1]:.8f}")
print(f"         f = {f_ill(*path_mom_ill[-1]):.10f}")
print(f"         距最优距离 = {np.sqrt(path_mom_ill[-1,0]**2 + path_mom_ill[-1,1]**2):.6f}")
print("-" * 70)

# 计算并打印损失比和距离比
print(f"损失比: f_GD / f_mom = {f_ill(*path_gd_ill[-1]) / f_ill(*path_mom_ill[-1]):.1f}×")
print(f"距离比: d_GD / d_mom = {np.sqrt(path_gd_ill[-1,0]**2 + path_gd_ill[-1,1]**2) / np.sqrt(path_mom_ill[-1,0]**2 + path_mom_ill[-1,1]**2):.1f}×")
print("=" * 70)

# 打印验证结果一：沿低曲率方向加速收敛
print("【验证结果一】沿低曲率方向加速收敛（4.2节）：")
print("  · x方向（平缓，梯度小）：HB法速度持续累积，")
print(f"    有效步长 ≈ lr/(1-β) = {lr_ill}/(1-{beta_ill}) = {lr_ill/(1-beta_ill):.3f}，")
print(f"    是GD法有效步长({lr_ill})的 {1/(1-beta_ill):.0f} 倍")
print("  · HB法在x方向前进距离远大于GD法，验证了加速收敛能力")
print()

# 打印验证结果二：高曲率方向抑制振荡
print("【验证结果二】高曲率方向抑制振荡（4.3节）：")
print("  · y方向（陡峭，梯度大）：HB法梯度符号交替时速度相互抵消，")
print("    振荡被抑制，不会发散")
print("  · 两者在y方向均保持稳定，验证了HB法的阻尼作用")
print("=" * 70)

【病态二次函数 κ=100：GD法 vs HB法】
相同学习率 α=0.018，相同步数 15
起点: (-8.0, 0.1), f=32.5000
----------------------------------------------------------------------
GD法  终点: x=-6.09203379, y=-0.00351844
         f = 18.5570568426
         距最优距离 = 6.092035
----------------------------------------------------------------------
HB法  终点: x=0.04439839, y=-0.00363457
         f = 0.0016461139
         距最优距离 = 0.044547
----------------------------------------------------------------------
损失比: f_GD / f_mom = 11273.3×
距离比: d_GD / d_mom = 136.8×
【验证结果一】沿低曲率方向加速收敛（4.2节）：
  · x方向（平缓，梯度小）：HB法速度持续累积，
    有效步长 ≈ lr/(1-β) = 0.018/(1-0.9) = 0.180，
    是GD法有效步长(0.018)的 10 倍
  · HB法在x方向前进距离远大于GD法，验证了加速收敛能力

【验证结果二】高曲率方向抑制振荡（4.3节）：
  · y方向（陡峭，梯度大）：HB法梯度符号交替时速度相互抵消，
    振荡被抑制，不会发散
  · 两者在y方向均保持稳定，验证了HB法的阻尼作用


#### 4.4.1 实验核心目的

本实验通过一个精心设计的二维病态二次函数，**同时定量验证**HB法的两个核心优势：

1. **沿低曲率方向加速收敛**（4.2节）——通过x方向展示
2. **高曲率方向抑制振荡**（4.3节）——通过y方向展示

#### 4.4.2 函数选择的深层理由

**1. 病态条件的代表性与普遍性**

深度学习中的损失函数通常具有极高的病态性——不同参数方向上的曲率差异可达数个数量级。本实验选取的条件数 $\kappa = 100$ 已足以清晰展示两种算法的行为差异。

**2. 二次函数的可解性与可解释性**

二次函数 $f(x,y) = \frac{1}{2}(x^2 + 100y^2)$ 是凸函数中最简单的形式：
- Hessian矩阵为常数：$H = \text{diag}(1, 100)$，分析干净纯粹
- 梯度有解析表达式：$\nabla f = (x, 100y)$，便于逐步追踪
- 收敛行为可精确预测

**3. 方向分离的巧妙设计**

函数中x和y方向的Hessian特征值分别为1和100，创造出两个截然不同的方向：

| 方向 | Hessian特征值 | 曲率 | 梯度幅值 | GD法行为 | HB法行为 | 验证的优势 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **x方向** | $\lambda_{\min}=1$ | 低（平缓） | 小（$=x$） | 步进极小，收敛瓶颈 | 速度累积，有效步长放大10倍 | **4.2节：加速收敛** |
| **y方向** | $\lambda_{\max}=100$ | 高（陡峭） | 大（$=100y$） | 步进大，容易振荡 | 振荡被抑制，不威胁稳定性 | **4.3节：抑制振荡** |

这种**方向分离**使得我们可以独立观察HB法在每个方向上的行为机制。

**4. 起点与步长的精心设置**

- **起点选择 $(-8, 0.1)$**：x坐标远离最优（-8），y坐标接近最优（0.1）。这样设置是为了**放大x方向（平缓方向）的收敛瓶颈**，同时让y方向初始时接近稳定。

- **学习率选择 $lr=0.018$**：
  - y方向的稳定步长上限为 $2/\lambda_{\max} = 0.02$
  - 取 $lr=0.018$ 略低于此上限，保证GD法在y方向**不会发散**
  - 这个学习率对x方向来说**非常小**——仅为x方向稳定上限 $2/\lambda_{\min} = 2$ 的0.9%

- **相同学习率、相同步数的公平对比**：确保性能差异**唯一归因于动量机制本身**

**5. 实验同时验证两个优势的逻辑**

GD法在病态条件下的困境源于：**学习率受到最陡峭方向的约束**。为了在y方向不振荡发散，必须将$lr$设得很小；但这样的小$lr$导致在x方向上每一步只能前进微小距离。

HB法通过**速度累积机制**同时解决了这两个问题：
- **x方向（梯度符号一致）**：速度持续累积 → 有效步长放大10倍 → **验证4.2节**
- **y方向（梯度符号交替）**：速度相互抵消 → 振荡被抑制 → **验证4.3节**

**6. 实验的预期结果**

经过15步迭代后：
- **GD法**：x方向仅前进很小距离，x仍远离0
- **HB法**：x方向前进距离远大于GD法，x显著更接近0
- 两者在y方向均保持稳定（不振荡发散）

这将**同时证明**：HB法既能在平缓方向加速，又能在陡峭方向保持稳定。

### 4.5 核心优势三：跨越鞍点与浅谷

高维优化中，**鞍点**是远比局部极小更常见的障碍。理解HB法如何跨越鞍点与浅谷，是理解其在深度学习中强大性能的关键。

#### 4.5.1 鞍点与浅谷：非凸优化的两大陷阱

**鞍点（Saddle Point）** 是一阶导数为零但既非局部极大也非局部极小的点。其Hessian矩阵的特征值有正有负：

- 沿某些方向，函数值是**上凸**的（正曲率，像山谷）
- 沿另一些方向，函数值是**下凸**的（负曲率，像山脊）
- 从整体看，它像一个**马鞍**：在一个方向上是"最低点"，在另一个方向上是"最高点"

**浅谷（Shallow Local Minimum）** 是局部极小值中"坑"较浅的那些。虽然梯度同样为零，但函数值仅比周围略低，远不是全局最优。

两者的共同特征是：**梯度 $\nabla f \approx \mathbf{0}$**。这使得以梯度为唯一导航信号的GD法在这些区域**寸步难行**。

#### 4.5.2 HB法跨越鞍点与浅谷的机制

HB法跨越鞍点与浅谷的核心机制可以概括为：**即使当前梯度消失，历史梯度的累积仍然推动参数继续前进**。

具体而言，动量更新公式（重球法HB）：

$$
v_{t+1} = \beta v_t + \alpha \nabla f(x_t)
$$

当 $\nabla f(x_t) \approx 0$（鞍点或浅谷底部）时：

$$
v_{t+1} \approx \beta v_t
$$

速度并不会立即消失，而是以 $\beta$ 的比例衰减。这意味着：

| 迭代步数 | 速度大小（相对初始速度 $v_0$） |
| :--- | :--- |
| 0 | $v_0$（进入鞍点时的速度） |
| 1 | $\beta v_0$ |
| 5 | $\beta^5 v_0$ |
| 10 | $\beta^{10} v_0$ |
| $k$ | $\beta^k v_0$ |

只要 $\beta$ 不太小，速度在若干步内仍然保持可观的值，足以推动参数**穿过**梯度为零的区域。

**对比GD法**：当 $\nabla f \approx 0$ 时，更新量 $\Delta x = -\alpha \nabla f \approx 0$，参数**立即停滞**，没有任何机制可以推动它继续前进。

#### 4.5.3 第1节算例正是本优势的直接验证

以上实验：

$$
f(x, y) = 0.2(x^2 - 1)^2 + y^2 - 0.15x
$$

该函数的三个一阶驻点构成了一条完整的"逃逸链"：

$$
\underbrace{(-0.8882, 0)}_{\text{局部极小}} \xrightarrow{\text{须跨越}} \underbrace{(-0.1949, 0)}_{\text{鞍点}} \xrightarrow{\text{须跨越}} \underbrace{(1.0831, 0)}_{\text{全局极小}}
$$

**GD法的行为**：

从起点 $(-0.1949, 1.0)$ 出发后，迅速滑入鞍点 $(-0.1949, 0)$，此时 $\nabla f \approx 0$，更新量趋近于零，**连第一个障碍都无法突破**，更不可能到达全局最优。

**HB法的行为**：

从同一起点出发，凭借累积的速度惯性：

1. **冲过鞍点**：滑向鞍点的过程中速度持续累积，到达鞍点时梯度虽已消失，但速度 $v_t$ 仍然保留，参数被惯性**推出鞍点区域**
2. **穿越稳定流形**：继续向右移动，穿越稳定流形 $x = -0.1949$（关于稳定流形的详细讨论见**附录K**）
3. **抵达全局最优**：越过稳定流形后，梯度重新出现并引导参数滑入全局极小 $(1.0831, 0)$

**定量对比**：

| 指标 | GD法 | HB法 |
| :--- | :--- | :--- |
| 最终位置 | $(-0.1949, 0)$ 鞍点 | $(1.0831, 0)$ 全局极小 |
| 最终损失 | $\approx 0.094$ | $\approx -0.316$ |
| 是否跨越鞍点 | 否（停滞） | 是 |
| 是否跨越稳定流形 | 否（未到达） | 是 |

该算例**同时展示了两种逃逸能力**，形成了完整的"**鞍点 → 稳定流形 → 全局极小**"逃逸链，是重球法逃逸能力的经典范例。

> **本节算例对应 4.1 节全景概览中的哪几条？**
>
> 本算例的目标函数是**非凸函数**，因此**不满足**"平方根加速"的理论保证条件。但它**直接验证了 4.1 节中的两条实践优势**：
>
> | 4.1 节条目 | 本算例中的体现 |
> | :--- | :--- |
> | **鞍点逃逸** ✅ | HB 法穿越了 $(-0.1949, 0)$ 处的鞍点，GD 法连到达鞍点的机会都没有 |
> | **稳定流形穿越** ✅ | HB 法穿越了 $x = -0.1949$ 处的稳定流形，GD 法则被困在鞍点处 |

#### 4.5.4 逃逸能力的关键因素

**动量系数 $\beta$ 的大小**：

- $\beta$ 越大 → 速度衰减越慢 → 惯性越持久 → 逃逸能力越强
- 第1节算例使用 $\beta=0.94$，对应速度衰减到一半约需 $\frac{\ln 0.5}{\ln 0.94} \approx 11$ 步

**进入陷阱时的速度大小**：

- 速度越大 → 冲过陷阱的概率越高
- 速度大小取决于此前路径上的梯度累积

**陷阱的"深度"**：

- 浅谷：宽度小、深度浅，容易冲过
- 深谷：宽度大、深度深，即使有动量也可能被困住

**代价与权衡**：

- $\beta$ 过大可能**冲过头**：连全局最优也冲过去，导致收敛末期反复震荡
- 这是重球法的一个固有局限，Nesterov动量通过"预判减速"缓解了这一问题（将在另一教程中讲解）

#### 4.5.5 高维空间中的鞍点问题

在深度学习中，鞍点问题比低维情形**严重得多**：

- **维度灾难**：$n$ 维空间中，随机临界点是鞍点而非局部极小的概率随 $n$ 指数趋近于1
- **鞍点高原**：深度网络的损失函数中存在大量**鞍点高原**（saddle plateau），梯度在广阔区域内接近零
- **梯度消失**：深层网络的梯度在反向传播中逐层衰减，导致大量参数方向的"有效梯度"极其微小

重球法在这些场景下的逃逸能力，是它成为深度学习基础优化器的核心原因之一。

### 4.6 局限性与注意事项

1. **超参数敏感**：$\beta$ 和学习率 $\alpha$ 需要配合调节。学习率过大或 $\beta$ 过高可能导致发散或振荡加剧。

2. **过冲风险**：在曲率突变区域，累积的动量可能导致**显著过冲**，使参数越过最优点，甚至反复弹跳。

3. **非自适应**：纯动量GD不考虑各参数方向上梯度的历史方差。对于极度稀疏或尺度差异巨大的参数，自适应方法（如Adam）可能更合适。

4. **理论加速需条件**：$O(\sqrt{\kappa})$ 的加速保证**仅对强凸二次函数严格成立**。对一般强凸光滑函数，固定 $\alpha, \beta$ 的HB没有全局加速保证，甚至可能不收敛；对非强凸或非凸问题，更没有完备的加速理论。实际深度学习中的加速效果主要是经验性的。详细分析见**附录I**。

## 5. 算法形式与深度理解

### 5.1 各分量含义

| 符号 | 名称 | 含义 | 典型值 |
| :--- | :--- | :--- | :--- |
| $x_t$ | 参数 | 当前位置 | — |
| $v_t$ | 速度/动量 | 参数变化的累积量 | 初始为0 |
| $\alpha$ (lr) | 学习率 | 梯度对速度的贡献强度 | 0.01 ~ 0.1 |
| $\beta$ | 衰减率 | 上一时刻速度的保留比例 | 0.9 |
| $\nabla f(x_t)$ | 梯度 | 当前位置的最速下降方向 | — |

### 5.2 展开视角：速度是历史梯度的加权和

将 $v_t$ 递归展开：

$$
v_{t+1} = -\alpha \sum_{i=0}^{t} \beta^{t-i} \nabla f(x_i)
$$

**核心洞察**：

- 当前速度是**所有历史梯度的指数加权平均**
- 越近的梯度权重越大（权重 $\propto \beta^{t-i}$）
- 这就是"动量"的本质——速度在持续积累历史信息

### 5.3 几何理解

$$
\text{新速度方向} = \underbrace{\beta v_t}_{\text{原方向/惯性}} + \underbrace{\alpha(-\nabla f)}_{\text{梯度方向/修正}}
$$

这是一个**向量加法**：

- GD法能急转弯：每一步完全听从当前梯度
- HB法不能急转弯：新方向是原方向和梯度方向的**妥协**
- 在峡谷中：横向的梯度震荡被抵消，纵向的速度得以积累

## 6. β 和 α 的深度解析

### 6.1 两个维度

| 维度 | 由谁决定 | 控制什么 |
| :--- | :--- | :--- |
| **绝对大小** | $\alpha$ 和 $\beta$ 共同决定（有效步长 $\frac{\alpha}{1-\beta}$） | **走多远**（收敛速度） |
| **相对大小** | $\frac{\beta\|v_t\|}{\alpha\|\nabla f\|}$（动态变化） | **往哪走**（方向选择） |

### 6.2 β 和 α 的职责分工

| 参数 | 职责 | 调大的效果 | 调小的效果 |
| :--- | :--- | :--- | :--- |
| $\beta$ | 控制记忆长度（惯性） | 更依赖历史方向，路径更直 | 更依赖当前梯度，路径更灵活 |
| $\alpha$ | 控制响应强度（敏感度） | 对当前梯度反应更剧烈 | 对当前梯度反应更温和 |

### 6.3 四象限组合分析

| | $\beta$ 大 (0.9~0.99) | $\beta$ 小 (0~0.5) |
| :--- | :--- | :--- |
| **$\alpha$ 大 (0.1~1.0)** | 两者都强：速度大但可能振荡 ⚠️ | 梯度主导：能急转弯，接近GD法 🔄 |
| **$\alpha$ 小 (0.001~0.01)** | 惯性主导：平滑稳定，动量典型 ✅ | 两者都弱：收敛慢 ❌ |

### 6.4 β 和 α 的关系

> **重要澄清**：$\beta$ 和 $\alpha$ 在数学定义上是独立的（可以任意赋值），但在算法行为中**高度耦合**。
>
> - $\beta$ 调大 → 有效步长 $\frac{\alpha}{1-\beta}$ 自动放大 → 需要调小 $\alpha$ 来补偿
> - 常用组合：$(\beta=0.9, \alpha=0.1)$ 和 $(\beta=0.99, \alpha=0.01)$ 具有相同的有效步长 ≈ 1.0
>
> **禁止规定 $\beta + \alpha = 1$**：这会破坏速度的积累能力，让动量退化为普通的低通滤波器，失去"冲刺"功能。

## 7. 实用调参策略

### 7.1 系统化调参方法

**步骤1：固定 $\beta$**

- 深度学习默认：$\beta = 0.9$
- 想要更平滑（强惯性）：$\beta = 0.99$
- 想要更灵活（弱惯性）：$\beta = 0.5$

**步骤2：调整 $\alpha$（学习率）**

- 从 $\alpha = 0.01$ 开始
- 如果Loss发散（梯度爆炸），减小 $\alpha$
- 如果收敛太慢（训练停滞），增大 $\alpha$

**步骤3：检查平衡**

经验公式：$\alpha \times \frac{1}{1-\beta}$ 应保持在合理范围（如 0.01 ~ 1.0）。

| 组合 | 检查 | 判断 |
| :--- | :--- | :--- |
| $\beta=0.9, \alpha=0.1$ | $0.1 \times 10 = 1.0$ | ✅ 平衡良好 |
| $\beta=0.99, \alpha=0.01$ | $0.01 \times 100 = 1.0$ | ✅ 平衡良好 |
| $\beta=0.9, \alpha=0.001$ | $0.001 \times 10 = 0.01$ | ⚠️ 收敛可能很慢 |
| $\beta=0.5, \alpha=0.1$ | $0.1 \times 2 = 0.2$ | ⚠️ 惯性较弱，可能震荡 |

### 7.2 学习率调度

动量法常与学习率调度策略配合使用：

- **阶梯衰减（Step Decay）**：每N个epoch将lr乘以0.1
- **余弦退火（Cosine Annealing）**：lr按余弦曲线平滑下降
- **Warmup**：训练初期先用小lr逐步增大，避免初始震荡

### 7.3 与其他优化器的对比

| 方法 | 收敛加速 | 振荡抑制 | 鞍点逃逸 | 自适应学习率 |
| :--- | :--- | :--- | :--- | :--- |
| GD法 | 无 | 差 | 弱 | 无 |
| HB法（重球法） | 好（强凸二次） | 好 | 较好 | 无 |
| Nesterov加速GD（NAG） | 最优（凸） | 好 | 较好 | 无 |
| AdaGrad/RMSProp | 中等 | 好 | 中等 | 有 |
| Adam | 好 | 好 | 好 | 有 |

**与自适应方法的互补关系**：
- HB法通过一阶矩估计（梯度均值）实现方向平滑和加速
- 自适应方法（RMSprop/Adam）通过二阶矩估计（梯度平方均值）实现逐参数学习率调整
- Adam = 动量（一阶矩）+ 自适应（二阶矩）的融合

### 7.4 后续发展

动量法的指数加权平均思想直接影响并催生了后续的自适应优化器：

- **RMSprop**：对梯度平方做EMA，实现自适应学习率
- **Adam**：同时维护一阶动量（梯度均值）和二阶动量（梯度平方均值）
- **NAdam**：将Nesterov预判机制融入Adam

> 📌 **最终结论**：重球法是一阶优化方法发展史上的里程碑，它以极小的计算代价（仅需额外存储一份速度向量）换来了收敛速度的质变。理解重球法，是深入理解所有现代优化器的基础。

---

## 附录

### 附录A：目标函数驻点分析（对应第1节）


目标函数：
$$f(x,y)=0.2(x^2-1)^2+y^2-0.15x$$

**求驻点（梯度条件 $\nabla f = \boldsymbol 0$）**
 一阶偏导数
对 $x$ 求偏导：
$$
\begin{aligned}
\frac{\partial f}{\partial x}
&= 0.2\cdot 2(x^2-1)\cdot 2x -0.15 \\
&=0.8x(x^2-1)-0.15
\end{aligned}
$$

对 $y$ 求偏导：
$$
\frac{\partial f}{\partial y}=2y
$$

 驻点方程组
$$
\begin{cases}
0.8x(x^2-1)-0.15 = 0 \\
2y = 0
\end{cases}
$$

由 $2y=0$，直接得到 $\boldsymbol{y=0}$，只需解一元三次方程：
$$0.8x^3 -0.8x -0.15 =0$$
两边放大20倍消去小数：
$$16x^3 -16x -3 = 0$$

三次方程的3个实根数值解：
- $x_1 \approx -0.8882$
- $x_2 \approx -0.1949$
- $x_3 \approx 1.0831$

得到全部驻点：
$$
P_1(-0.8882,\ 0),\quad P_2(-0.1949,\ 0),\quad P_3(1.0831,\ 0)
$$

**Hessian矩阵判别极值类型**
二阶偏导数：
$$
\frac{\partial^2 f}{\partial x^2}=0.8(3x^2-1),\quad
\frac{\partial^2 f}{\partial x\partial y}=0,\quad
\frac{\partial^2 f}{\partial y^2}=2
$$

Hessian矩阵：
$$
H(x)=
\begin{pmatrix}
0.8(3x^2-1) & 0 \\
0 & 2
\end{pmatrix}
$$

Hessian行列式：
$$
\det(H)=1.6(3x^2-1)
$$

各驻点分类
1. **$P_1(-0.8882,\ 0)$**
$\det(H)>0$，$H_{11}>0$，局部极小点，函数值 $f(P_1)\approx 0.1421$

1. **$P_2(-0.1949,\ 0)$**
$\det(H)<0$，鞍点，函数值 $f(P_2)\approx0.2143$

1. **$P_3(1.0831,\ 0)$**
$\det(H)>0$，$H_{11}>0$，局部极小点；同时为全局极小点，函数值 $f(P_3)\approx \boldsymbol{-0.1565}$

> 全局极小说明：$f(x,y)=g(x)+y^2$，$y^2\ge0$，最小值必然出现在 $y=0$；$x\to\pm\infty$ 时函数趋向正无穷，比较两个局部极小，$P_3$ 为全局最小。

**迭代起点到各驻点的欧氏距离**
迭代寻优起点：$\boldsymbol S = (-0.1949,\ 1.0000)$

欧氏距离公式：
$$d(S,P)=\sqrt{(x_S-x_P)^2+(y_S-y_P)^2}$$

距离计算
1. 起点 $\to P_1(-0.8882,0)$
$$d_1=\sqrt{(-0.1949+0.8882)^2+(1.0000-0)^2} \approx 1.2168$$

1. 起点 $\to P_2(-0.1949,0)$
$$d_2=\sqrt{(-0.1949+0.1949)^2+(1.0000-0)^2}=1.0000$$

1. 起点 $\to P_3(1.0831,0)$
$$d_3=\sqrt{(-0.1949-1.0831)^2+(1.0000-0)^2}\approx1.6227$$

距离汇总表
|驻点|欧氏距离|
|---|---|
|$P_1(-0.8882,0)$|$\approx 1.2168$|
|$P_2(-0.1949,0)$|$1.0000$|
|$P_3(1.0831,0)$|$\approx 1.6227$|

> 备注：起点x坐标与鞍点$P_2$完全相等，仅y方向相差1，因此距离恰好等于1，起点在鞍点正上方。


### 附录B：条件数与病态条件问题

**条件数（Condition Number）** 是衡量优化问题病态程度的核心指标。

#### B.1 条件数的定义

对于二次函数 $f(x) = \frac{1}{2}x^T H x$（其中 $H$ 是Hessian矩阵），条件数定义为：

$$
\kappa = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

其中 $\lambda_{\max}$ 和 $\lambda_{\min}$ 分别是Hessian矩阵的最大和最小特征值。

#### B.2 条件数的影响

| 条件数 κ | 病态程度 | GD法收敛速度 |
| :--- | :--- | :--- |
| 1 | 理想（各方向曲率相同） | 最快 |
| 10 | 轻度病态 | 较慢 |
| 100 | 中度病态 | 很慢 |
| 1000+ | 重度病态 | 极慢 |
| $10^6$（深度学习常见） | 极度病态 | 几乎停滞 |

#### B.3 病态条件对GD法的影响

GD法在病态条件下的困境源于：**学习率受到最陡峭方向的约束**。

- 为了在陡峭方向不振荡发散，学习率必须设得很小
- 这么小的学习率在平缓方向上几乎走不动
- 导致整体收敛极慢，路径呈"之字形"震荡

### 附录C："病态函数"？

**"病态"（ill-conditioned）是对优化算法而言的**，不是函数本身有问题。

#### C.1 本例中的病态性

在 $f(x,y) = 0.5(x^2 + 100y^2)$ 中：

$$
H = \begin{bmatrix} 1 & 0 \\ 0 & 100 \end{bmatrix}
$$

- $\lambda_{\min} = 1$（x方向）
- $\lambda_{\max} = 100$（y方向）
- $\kappa = 100/1 = 100$

#### C.2 病态的表现

1. **学习率受限**：为了在陡峭方向（y）不振荡发散，学习率必须设得很小（$< 2/100 = 0.02$）

2. **收敛极慢**：这么小的学习率在平缓方向（x）上几乎走不动，导致整体收敛极慢

3. **等高线极度扁长**：函数的等高线是长短轴比为 $\sqrt{\kappa} = 10:1$ 的椭圆，梯度方向几乎垂直于最优方向

#### C.3 地形类比

想象一个**极度狭长的峡谷**：

- 横向（y方向）：谷壁陡峭，坡度大
- 纵向（x方向）：谷底平缓，坡度小

GD法在这种地形上会：
- 在横向来回弹跳（因为坡度大）
- 在纵向缓慢爬行（因为坡度小）
- 整体呈现"之字形"路径，效率极低

#### C.4 总结

**"病态"指的是优化问题的病态，而非函数本身的病态**。具体来说：

- 函数本身是"健康"的：光滑、凸、有唯一最优解
- 但Hessian矩阵的特征值差异巨大（κ=100）
- 导致梯度下降类算法在优化这个函数时"生病"：收敛极慢、路径震荡
- 就像一个人身体健康，但走在极度狭长的峡谷里会"走得很痛苦"

这也是为什么动量法、自适应学习率方法（如Adam）等改进算法如此重要——它们正是为了"治疗"这种病态问题而设计的。

### 附录D：光滑凸函数与强凸函数

**光滑凸函数**（$L$-smooth convex function）和**强凸函数**（$\mu$-strongly convex function）是分析优化算法收敛性的理论基石。
可参考：
https://zhuanlan.zhihu.com/p/369961290
https://zhuanlan.zhihu.com/p/210252556


#### D.1 凸函数基础

在讨论"光滑"和"强"凸性之前，必须先明确什么是凸函数。

**定义：**
设 $C \subseteq \mathbb{R}^n$ 是一个**凸集**。一个函数 $f: C \rightarrow \mathbb{R}$ 是凸函数，如果对于任意两个点 $x, y \in C$，以及任意 $\theta \in [0, 1]$，都满足：

$$
f(\theta x + (1-\theta) y) \le \theta f(x) + (1-\theta) f(y)
$$

> **为什么必须定义在凸集上？**
> 
> 凸函数定义的核心在于取定义域内任意两点的凸组合 $\theta x + (1-\theta) y$，然后比较函数值。**凸集**的定义恰好保证了这一操作的封闭性：对于任意 $x, y \in C$ 和任意 $\theta \in [0, 1]$，凸组合点 $\theta x + (1-\theta) y$ 仍然在 $C$ 中。如果定义域不是凸集，凸组合点可能落在定义域之外，导致 $f(\theta x + (1-\theta) y)$ 没有定义，不等式本身就无法验证。
>
> 在机器学习等实际场景中，损失函数通常定义在整个参数空间 $\mathbb{R}^n$ 上，而 $\mathbb{R}^n$ 本身就是最大的凸集，因此这一前提自动满足。但当定义域是某个受限集合时，必须首先确认该集合是凸集，才能讨论其上的凸函数。

**几何直觉：**
连接函数图像上任意两点 $(x, f(x))$ 和 $(y, f(y))$ 的线段，始终位于函数图像的上方或与之重合。换句话说，函数的"碗口"是朝上的。

**一阶条件（如果函数可微）：**
函数 $f$ 是凸的，当且仅当对于任意 $x, y \in C$，有：

$$
f(y) \ge f(x) + \nabla f(x)^T (y - x)
$$

这表示函数图像始终位于其任意一点的切平面（或切线）之上。切平面是函数的一个全局下估计。

**二阶条件（如果函数二阶可微）：**
函数 $f$ 是凸的，当且仅当其后森矩阵（Hessian）$\nabla^2 f(x)$ 对于所有 $x \in C$ 都是半正定的：

$$
\nabla^2 f(x) \succeq 0
$$

这意味着函数的曲率在所有方向上都是非负的。

#### D.2 光滑凸函数（L‑光滑 / 梯度L‑利普希茨连续）

光滑凸函数，是同时满足**凸性**与 **L‑光滑（梯度L‑利普希茨连续）** 的函数。

> 重要注记：
> **L‑光滑本身不依赖凸性**，非凸函数也可以是 L‑光滑的。
> 1. 二次上界、海森上界条件：L‑光滑自带性质，**不需要凸**
> 2. 余强制性：必须同时满足 **凸 + L‑光滑** 才成立

**1. L‑光滑 定义（不限制凸性）**

一个可微函数 $f$ 被称为 $L$-光滑的，如果其梯度 $\nabla f(x)$ 是 $L$-利普希茨连续的，即存在常数 $L>0$，对任意 $x,y$ 满足：
$$
\|\nabla f(x)-\nabla f(y)\|_2 \le L\|x-y\|_2
$$
$\|\cdot\|_2$ 为欧几里得距离；$L$ 称为光滑常数/利普希茨常数。

> 注：梯度L‑利普希茨条件也可以等价写成分式形式
> $$\frac{\|\nabla f(x)-\nabla f(y)\|_2}{\|x-y\|_2}\le L,\quad \forall x\neq y$$
> 表示梯度映射的变化率全局不超过 $L$；但标准教材一般写成 $\|\nabla f(x)-\nabla f(y)\|_2 \le L\|x-y\|_2$。优势是无需排除 $x=y$ 的情形，便于不等式代数推演。

**关键概念辨析：L‑光滑 VS 普通可导（高频易错）**

> 补充说明：日常工科语境常常把**$C^1$（一阶连续可导）**称作光滑，只要曲线无尖角、切线连续变化，视觉上就是光滑；
> - $C^1$：一阶导数存在且连续；
> - $C^\infty$（读作C‑无穷）：无穷阶连续可导，函数的任意阶导数均存在且连续，微分几何严格定义的光滑；
> - L‑光滑（优化定义）：条件强于 $C^1$，即使函数属于 $C^1$ 甚至 $C^\infty$，也未必L‑光滑，它额外约束梯度的变化速率存在全局上界。

> 注：L‑光滑是独立于 $C^k$ 体系的约束条件。
> $C^1/C^\infty$ 刻画**导数的阶数与连续性**；L‑光滑刻画**梯度映射的变化速率存在全局上界**。
> - L‑光滑 $\implies f\in C^1$；
> - $C^1$ 甚至 $C^\infty$ **推不出 L‑光滑**；
> L‑光滑是 $C^1$ 函数集合内部一个更加严苛的子集条件。

> 核心词义区分（极易混淆）：
> 1. 高等数学/工科直观语境："光滑"常指 $C^1$，曲线无尖角，切线连续变化；微分几何中"光滑"指 $C^\infty$ 无穷阶连续可导。
> 2. 凸优化/机器学习语境：本笔记的 **L‑光滑** 是完全独立的专业定义。

**普通可导（一阶可微）**
仅要求函数每一点**梯度存在、有切线**，**完全不限制梯度的变化速度**。
可导只保证"有导数"，不管"导数变得多猛"。

**L‑光滑（优化专属定义）**
是**比可导更强的条件**，必须同时满足两层：
1. 函数处处可导（梯度存在）；
2. 梯度自身满足L‑利普希茨连续，梯度变化速率存在**全局有限上界**。

**核心结论**
1. **L‑光滑 $\implies$ 一定可导，且属于 $C^1$**
2. **可导 $\nRightarrow$ L‑光滑；$C^1 \nRightarrow$ L‑光滑；$C^\infty \nRightarrow$ L‑光滑**

**经典反例印证**
$f(x)=x^{4/3}$、$f(x)=e^x$：
全域属于 $C^1$，$e^x$ 更是 $C^\infty$（高数意义上高度光滑），但在优化意义下**不是 L‑光滑**。
原因：二阶导数可以趋于无穷，梯度变化无界，不存在有限光滑常数 $L$。

**终极一句话总结**
- 普通可导：只看**有没有导数**；
- $C^1$：看导数是否连续，曲线视觉是否顺滑；
- L‑光滑：看**导数会不会剧烈乱跳，变化速率有没有全局上界**；
- 梯度下降收敛依赖的是 **L‑光滑**，不是普通可导，也不是 $C^1/C^\infty$。

**2. L‑光滑 核心等价性质（对任意函数，无需凸）**

**2.1 二次上界（充要条件）**
可微函数为 L‑光滑 $\iff$ 对任意 $x,y$：
$$
f(y)\le f(x)+\nabla f(x)^T(y-x)+\frac{L}{2}\|y-x\|_2^2
$$
含义：函数增长全局被二次函数压制，梯度变化不会无限剧烈。

**2.2 二阶可微条件（充要条件）**
若 $f$ 二阶连续可微，则 $L$-光滑等价于全局海森上有界：
$$
\nabla^2 f(x)\preceq LI,\quad \forall x
$$
含义：海森所有特征值有全局上界 $L$；**允许负特征值（允许非凸）**。
> ⚠️必须全局成立；局部有界不能推出全局 L‑光滑。

**3. 凸 + L‑光滑 联合性质（凸优化标准场景）**

**3.1 双边函数界**
叠加**凸函数下界**与 **L‑光滑上界**：
$$
f(x)+\nabla f(x)^T(y-x)\le f(y)\le f(x)+\nabla f(x)^T(y-x)+\frac{L}{2}\|y-x\|_2^2
$$

**3.2 二阶联合充要条件**
同时满足凸 + L‑光滑 $\iff$ 海森矩阵全局夹在 $0$ 和 $L$ 之间：
$$
0 \preceq \nabla^2 f(x) \preceq L I,\quad \forall x
$$
- 左半约束 $\nabla^2 f\succeq 0$：保证凸
- 右半约束 $\nabla^2 f\preceq LI$：保证 L‑光滑

**3.3 余强制性（Co‑coercivity）**
> ⚠️**仅 L‑光滑不够，必须凸 + L‑光滑**
梯度满足稳定性条件：
$$
\langle\nabla f(x)-\nabla f(y),x-y\rangle \ge \frac1L\|\nabla f(x)-\nabla f(y)\|_2^2
$$
是梯度下降收敛证明的核心工具。

**4. 几何直觉**
- **L‑光滑**：限制梯度变化速度，函数曲率存在**全局上界**，不会突然变陡。只约束梯度变化快慢，不约束曲率符号，因此可以是非凸函数。
- **凸 + L‑光滑**：像一个弯曲有限的碗，形状稳定、无无限弯折，优化性质极好。

**5. 凸且 L‑光滑 的标准例子**

- $f(x)=\frac12 x^2$：凸，1‑光滑
- $f(x) = \log(1 + e^x)$（逻辑回归损失）：凸，1‑光滑
- $f(x) = \|x\|_2^2$：凸，2‑光滑

> 补充推导：$f(x)=\|x\|_2^2 = x_1^2+x_2^2+\dots+x_n^2$
> 1. 梯度：$\nabla f(x)=2x$
> 2. 海森矩阵：$\nabla^2 f(x)=2I$，$I$为单位矩阵。
>
> 代入L‑光滑定义：
> $$
> \|\nabla f(x)-\nabla f(y)\|_2=\|2x-2y\|_2=2\|x-y\|_2
> $$
> 满足 $\|\nabla f(x)-\nabla f(y)\|_2 = 2\|x-y\|_2$，最小 $L=2$，故为2‑光滑。
>
> > 对比：$f(x)=\frac12\|x\|_2^2$，海森为 $I$，$L=1$，1‑光滑。

**6. 光滑但非凸（L‑光滑，但不凸）**

**6.1 概念说明**
**光滑但非凸**：梯度满足L‑利普希茨连续（存在有限$L$），海森特征值有全局上界，但海森矩阵不是半正定，存在负特征值，函数存在局部极小、鞍点。L‑光滑只限制曲率的最大绝对值，不禁止负曲率。

**6.2 典型例子**

> 例1：$f(x)=-\frac12 x^2$
> - 光滑：$f'(x)=-x$，$|f'(x)-f'(y)|=|x-y|$，$L=1$，1‑光滑；
> - 非凸：$f''(x)=-1<0$，全局凹函数。

> 例2：$f(x)=\cos(x),\ x\in\mathbb R$
> - 光滑：$f'(x)=-\sin x$，$|f'(x)-f'(y)|\le |x-y|$，$L=1$，1‑光滑；
> - 非凸：$f''(x)=-\cos x$，可取负值；既有波峰又有波谷，大量局部极大极小点。

> 例3：$f(x_1,x_2)= x_1^2 - x_2^2$（鞍点函数）
> - 光滑：海森 $\nabla^2 f=\begin{bmatrix}2&0\\0&-2\end{bmatrix}$，特征值为 $2,-2$，最大特征值2，$L=2$，2‑光滑；
> - 非凸：存在负特征值，原点为鞍点，不是凸函数。

**7. 凸但不 L‑光滑（重要反例对比）**

**7.1 概念说明**
**凸但不光滑**：函数满足凸性（海森 $\succeq 0$），但**梯度无全局 Lipschitz 常数**。
二阶可微等价描述：**海森特征值可以趋向无穷大**，曲率无上限、梯度变化可以无限剧烈。

**7.2 典型例子**

> 例1：$f(x)=x^{4/3},\;x\in\mathbb R$
> - 凸：$f''(x)=\dfrac49 x^{-2/3}\ge 0$，全局凸
> - 不光滑：$x\to0$ 时 $f''(x)\to+\infty$，无全局上界，**不存在有限 $L$**
> - 特点：属于 $C^1$，图像视觉光滑，但不是L‑光滑（关键易错点）

> 例2：$f(x)=e^x,\;x\in\mathbb R$
> - 凸：$f''(x)=e^x>0$，严格凸
> - 不光滑：$x\to+\infty,f''(x)\to+\infty$，全域无有限 $L$
> 补充：有限区间 $(-\infty,a]$ 上为 $e^a$-光滑，但**全域不光滑**

> 例3：$f(x)=|x|$
> - 凸：标准凸函数
> - 不光滑：原点不可微，梯度不唯一，**不满足L光滑前提（处处可微）**

**7.3 凸但非L‑光滑的共性**
1. 曲率/二阶导数可以趋于无穷，无全局上界
2. 梯度变化可无限剧烈，不满足 Lipschitz
3. 无法使用 L‑光滑 二次上界，标准梯度下降收敛理论不适用



**8. 四类函数终极对比表**

| 函数 | 凸？ | L‑光滑？ | 核心原因 |
|:---:|:---:|:---:|---|
| $\frac12 x^2$ | ✅ | ✅ | 海森恒定半正定，有上下界 |
| $x^{4/3}$ | ✅ | ❌ | $C^1$，可微但二阶导无上限 |
| $e^x$ | ✅ | ❌ | $C^\infty$，二阶导随 $x$ 指数爆炸 |
| $\lvert x \rvert$ | ✅ | ❌ | 原点不可微 |
| $-x^2$ | ❌ | ✅ | 海森有上界，特征值恒负，凹函数 |
| $\cos(x)$ | ❌ | ✅ | 海森可负，梯度变化有界 |
| $x_1^2-x_2^2$ | ❌ | ✅ | 海森含负特征值，存在鞍点 |

#### D.3 强凸函数（$\mu$-强凸）

强凸函数是比凸函数更强的性质。它不仅要求函数是凸的，还要求它"足够"凸，即它必须像一个二次函数一样向下弯曲，具有一个正的曲率下界。

**定义（$\mu$-强凸）：**
一个可微的凸函数 $f$ 被称为 $\mu$-强凸的，如果存在一个常数 $\mu > 0$，使得对于任意 $x, y$，都有：

$$
f(y) \ge f(x) + \nabla f(x)^T (y - x) + \frac{\mu}{2} \|y - x\|_2^2
$$

这里的 $\mu$ 被称为强凸性常数或强凸性模量。

**关键性质与等价条件：**

1. **二次下界：** 这个定义本身就给出了一个二次下界。它比普通凸性的一阶条件 $f(y) \ge f(x) + \nabla f(x)^T (y - x)$ 更强，多了一个正的二次项 $\frac{\mu}{2} \|y - x\|_2^2$。这意味着函数不仅在其切平面上方，而且与切平面之间还有一个二次函数的"间隙"。函数增长得比线性函数更快。

2. **二阶条件：** 如果 $f$ 是二阶可微的，那么 $\mu$-强凸性等价于其海森矩阵在所有点 $x$ 上都有一个正的下界：

$$
\nabla^2 f(x) \succeq \mu I
$$

这意味着海森矩阵的最小特征值至少是 $\mu$。

3. **与一个二次函数的关系：** $f(x) - \frac{\mu}{2}\|x\|_2^2$ 仍然是一个凸函数。这直观地展示了强凸函数可以被看作是一个普通凸函数和一个二次项之和。

**几何直觉：**
想象一个碗。强凸函数的"碗"具有一个最小的弯曲程度。它不能有任何平坦的区域。碗底是圆润的，而不是平的。$\mu$ 给出了这个碗的最小曲率的下限。一个普通的凸函数（如 $f(x) = x$ 或 $f(x) = \max(0, x)$）可能有平坦区域，因此不是强凸的。

**例子：**
- $f(x) = x^2$ 是 2-强凸的
- $f(x) = \frac{1}{2}x^2 + x$ 是 1-强凸的
- $f(x) = e^x$ 在区间 $[a, \infty)$ 上是 $e^a$-强凸的，但在整个实数域 $\mathbb{R}$ 上不是强凸的

> $m$-强凸 + $L$-光滑
> $$
> m I \preceq \nabla^2 f(x) \preceq L I,\quad \forall x
> $$
> $m>0$ 为强凸常数；条件数 $\kappa=\frac{L}{m}$，直接决定梯度下降收敛速率。
> 对应的双边函数界：
> $$
> f(x)+\nabla f(x)^T(y-x)+\frac m2\|y-x\|_2^2
> \;\le\; f(y)
> \;\le\; f(x)+\nabla f(x)^T(y-x)+\frac L2\|y-x\|_2^2
> $$

#### D.4 总结与对比

| 特性 | 凸函数 | 光滑凸函数 ($L$-光滑) | 强凸函数 ($\mu$-强凸) |
| :--- | :--- | :--- | :--- |
| **核心思想** | 碗口朝上 | 碗口朝上，且曲率有上限 | 碗口朝上，且曲率有正的下限 |
| **一阶条件** | $f(y) \ge f(x) + \nabla f(x)^T(y-x)$ | (无直接对应) | $f(y) \ge f(x) + \nabla f(x)^T(y-x) + \frac{\mu}{2}\|y-x\|^2$ |
| **二阶条件** | $\nabla^2 f(x) \succeq 0$ | $\nabla^2 f(x) \preceq L I$ | $\nabla^2 f(x) \succeq \mu I$ |
| **关键不等式** | 线性下界 | 二次上界 | 二次下界 |
| **梯度性质** | 单调性 | 利普希茨连续（变化有界） | 强单调性 |
| **几何图像** | 任意凸的碗 | 内壁平滑、曲率有限的碗 | 底部圆润、无平坦区域的碗 |
| **典型例子** | $f(x) = \|x\|$ | $f(x) = \frac{1}{2}\|x\|^2$ 的缩放版 | $f(x) = \frac{1}{2}\|x\|^2$ |
| **对优化的意义** | 保证全局最优解 | 保证梯度下降等算法的收敛性 | 保证算法的线性收敛速率，且最优解唯一 |

#### D.5 强凸 + 光滑：条件数与优化难度

当一个函数同时是 $\mu$-强凸和 $L$-光滑时，它被很好地"夹在"两个二次函数之间：

$$
\frac{\mu}{2}\|y-x\|^2 \le f(y) - f(x) - \nabla f(x)^T(y-x) \le \frac{L}{2}\|y-x\|^2
$$

其海森矩阵满足 $\mu I \preceq \nabla^2 f(x) \preceq L I$。

**条件数：**
比值 $\kappa = L/\mu$ 被称为函数的条件数。它衡量了函数"碗"的细长程度或各向异性。

- $\kappa$ 越大，碗越细长，优化问题就越困难
- $\kappa$ 越接近 1，碗越接近球形，优化问题就越容易
- 梯度下降法等一阶方法的收敛速度通常与 $\kappa$ 密切相关

**与正文的联系：**
- GD法在强凸光滑函数上的收敛速率为 $O(\kappa \log(1/\epsilon))$
- HB法（最优动量）**在强凸二次函数上**为 $O(\sqrt{\kappa} \log(1/\epsilon))$
- 这就是重球法在病态条件（$\kappa$ 大）下实现"平方根加速"的理论来源

> **注意**：上述 $O(\sqrt{\kappa})$ 加速速率对 HB 而言**仅在强凸二次函数上有严格保证**。对一般强凸光滑函数，NAG 才有 $O(\sqrt{\kappa})$ 的全局加速保证。详见附录I。

#### D.6 在机器学习中的应用

在机器学习中，我们经常遇到带有正则化项的损失函数：

- **L2 正则化（岭回归）：** 在凸损失函数（如线性回归的均方误差）上添加 $\frac{\lambda}{2}\|w\|^2$ 项，可以使整个目标函数变为 $\lambda$-强凸的。这不仅有助于防止过拟合，还保证了优化问题有唯一解，并且加速了优化算法的收敛。

- **光滑性：** 许多损失函数（如逻辑回归、带 Huber 损失的回归）本身是光滑的，这使得梯度下降及其变体能够很好地工作。

理解这两个概念，是深入学习和分析优化算法收敛性、设计更有效算法的基石。

### 附录E：步长安全区间与学习率鲁棒性

对于 $L$-Lipschitz 连续梯度的凸函数：

| 算法 | 收敛条件 | 典型上限（$L=1$） |
| :--- | :--- | :--- |
| GD法 | $0 < \alpha < 2/L$ | 必须小于 2.0 |
| HB法 | $0 < \alpha < 2(1+\beta)/L$ | 若 $\beta=0.9$，上限为 **3.8** |

**关键结论**：重球法的步长安全区间比GD法宽了近一倍，因此**重球法对学习率更鲁棒**。

**有效步长**：

$$
\text{有效步长} \approx \frac{\alpha}{1-\beta}
$$

| 组合 | 有效步长 | 行为特征 |
| :--- | :--- | :--- |
| $\alpha=0.01, \beta=0.9$ | 0.1 | 平稳收敛（常用） |
| $\alpha=0.1, \beta=0.9$ | 1.0 | 快速但可能剧烈 |
| $\alpha=0.01, \beta=0.99$ | 1.0 | 极其平滑但步长远大于 $\alpha$ |

> **实战启示**：调大 $\beta$ 后，如果不相应调小 $\alpha$，有效步长会自动放大，可能导致Loss爆炸。

### 附录F：动量系数 $\beta$ 对寻优性能的影响

| $\beta$ 取值 | 有效记忆长度 | 行为特征 |
| :--- | :--- | :--- |
| 0 | 1步 | 退化为GD法，无惯性 |
| 0.5~0.8 | 2~5步 | 温和加速，振荡抑制一般 |
| 0.9 | ~10步 | **经典选择**，综合性能最佳 |
| 0.99 | ~100步 | 极强惯性，适合极平滑方向加速，但可能过冲、不稳定 |

**惯性强度与逃逸能力的权衡**：
- $\beta$ 越大 → 记忆越长 → 逃逸鞍点和浅谷的能力越强
- $\beta$ 越大 → 惯性越强 → 过冲风险越大，收敛末期振荡可能加剧

### 附录G：收敛性证明（强凸二次函数情形）

**在特定条件下HB法收敛显著快于GD法**

---

**条件一：目标函数是二次函数**

$$
f(x) = \frac{1}{2}x^T A x - b^T x
$$

- 损失地形的形状固定，是完美的"碗状"或"狭长峡谷状"
- Hessian矩阵 $A$ 恒定不变
- 没有深度学习损失函数中常见的坑坑洼洼、鞍点、局部极小等复杂结构

---

**条件二：超参数调到理论最优**

**GD法的最优学习率：**

$$
\eta^* = \frac{2}{\lambda_1 + \lambda_n}
$$

**HB法的最优学习率和动量系数：**

$$
\eta^* = \frac{4}{(\sqrt{\lambda_n} + \sqrt{\lambda_1})^2}
$$

$$
\mu^* = \left(\frac{\sqrt{\lambda_n} - \sqrt{\lambda_1}}{\sqrt{\lambda_n} + \sqrt{\lambda_1}}\right)^2
$$

- 两者都必须在各自的理论最优超参数下比较
- 实际中随意调参时，GD法完全可能比调参糟糕的HB法更快

---

**条件三：问题病态**

- 不同方向曲率差异很大
- 条件数 $\kappa = \frac{\lambda_{\max}}{\lambda_{\min}}$ 很大
- 函数像一个又窄又长的峡谷

**对比：**

| 条件数 $\kappa$ | GD法收敛速率 | HB法收敛速率 | 加速效果 |
|:---:|:---:|:---:|:---:|
| 接近 1 | $O(\kappa)$ | $O(\sqrt{\kappa})$ | 几乎无差异 |
| 很大 | $O(\kappa)$ | $O(\sqrt{\kappa})$ | **显著加速** |

---

**总结**

> **问题越病态（$\kappa$ 越大），HB法的优势越明显。**
>
> 在 $\kappa$ 接近 1 的"圆碗"地形上，两者速度几乎一样；
> 在 $\kappa$ 很大的"狭长峡谷"地形上，HB法实现平方根加速。

本附录给出正文4.6节中收敛速率的严格数学推导。核心思路是：将算法迭代转化为误差向量的**差分方程**，通过**特征分解**解耦，再求解每个方向上的**收敛因子**。

> **适用范围声明**：以下推导**仅对强凸二次函数** $f(x) = \frac{1}{2}x^T A x - b^T x$（$A \succ 0$）严格成立。对一般强凸光滑函数，HB 没有相应的全局加速保证。详见附录I。

#### G.1 问题设定与误差递推

考虑二次目标函数 $f(x) = \frac{1}{2} x^T A x - b^T x$，其中 $A$ 是对称正定矩阵。

**最优解条件**：
$$
\nabla f(x^*) = 0 \quad \Rightarrow \quad b = A x^*
$$

**定义误差向量** $e_k = x_k - x^*$。

**梯度转化**：
$$
\nabla f(x_k) = A x_k - b = A(x_k - x^*) = A e_k
$$

**GD法的误差递推**：
$$
e_{k+1} = x_{k+1} - x^* = x_k - \eta \nabla f(x_k) - x^* = e_k - \eta A e_k = (I - \eta A) e_k
$$

**HB法的误差递推**：
HB法的速度形式为：
$$
v_{k+1} = \mu v_k - \eta \nabla f(x_k), \quad x_{k+1} = x_k + v_{k+1}
$$
消去 $v$ 得到：
$$
x_{k+1} = x_k + \mu(x_k - x_{k-1}) - \eta \nabla f(x_k)
$$
代入 $\nabla f(x_k) = A e_k$ 并整理为误差形式：
$$
e_{k+1} - (1 + \mu - \eta A) e_k + \mu e_{k-1} = 0
$$

#### G.2 特征分解与解耦

对对称矩阵 $A$ 进行特征分解 $A = Q \Lambda Q^T$，其中 $\Lambda = \text{diag}(\lambda_1, \dots, \lambda_n)$，$Q$ 为正交矩阵。令 $y_k = Q^T e_k$。

由于 $Q$ 是正交的，$e_k$ 的收敛行为与 $y_k$ 完全相同。两边同乘 $Q^T$，误差递推解耦为 $n$ 个独立的标量方程：

- **GD法**：
$$
y_{k+1} = (1 - \eta \lambda_i) y_k
$$
其通项为 $y_k = (1 - \eta \lambda_i)^k y_0$。

- **HB法**：
$$
y_{k+1} - (1 + \mu - \eta \lambda_i) y_k + \mu y_{k-1} = 0
$$
这是一个二阶差分方程。设 $y_k = \xi^k$，得到特征方程：
$$
\xi^2 - (1 + \mu - \eta \lambda_i) \xi + \mu = 0
$$
其通项为 $y_k = C_1 \xi_1^k + C_2 \xi_2^k$，其中 $\xi_1, \xi_2$ 是特征方程的两个根。

#### G.3 收敛因子分析

收敛速率由 $y_k$ 通项公式中决定衰减速度的部分决定，即**收敛因子**。

**GD法的收敛因子**：
$$
r_{GD}(\lambda_i) = |1 - \eta \lambda_i|
$$
为保证收敛，必须对所有特征值 $\lambda_i$ 都有 $|1 - \eta \lambda_i| < 1$，这要求 $0 < \eta < 2/\lambda_{\max}$。

**最优学习率**：最小化最大收敛因子 $\min_\eta \max_i |1 - \eta \lambda_i|$。最优策略是均衡化，使收敛因子在 $\lambda_1$（最小）和 $\lambda_n$（最大）处相等：
$$
1 - \eta \lambda_1 = \eta \lambda_n - 1 \quad \Rightarrow \quad \eta^* = \frac{2}{\lambda_1 + \lambda_n}
$$
此时收敛因子为：
$$
r_{GD}^* = \frac{\lambda_n - \lambda_1}{\lambda_n + \lambda_1} = \frac{\kappa - 1}{\kappa + 1} \approx 1 - \frac{2}{\kappa}
$$
其中 $\kappa = \lambda_n / \lambda_1$ 为条件数。

**HB法的收敛因子**：
$$
r_M(\lambda_i) = \max\{|\xi_1|, |\xi_2|\}
$$
最优策略是使特征方程两根重合（**临界阻尼**），并对 $\lambda_1$ 和 $\lambda_n$ 同时达到临界阻尼。求解得到最优超参数：
$$
\sqrt{\mu^*} = \frac{\sqrt{\lambda_n} - \sqrt{\lambda_1}}{\sqrt{\lambda_n} + \sqrt{\lambda_1}}, \quad \eta^* = \frac{4}{(\sqrt{\lambda_n} + \sqrt{\lambda_1})^2}
$$
此时收敛因子为：
$$
r_M^* = \frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1} \approx 1 - \frac{2}{\sqrt{\kappa}}
$$

#### G.4 结论：重球法的平方根加速

在低曲率方向（$\lambda = \lambda_1$）上比较两者最优收敛因子：
$$
r_M^* \approx 1 - \frac{2}{\sqrt{\kappa}} \quad < \quad r_{GD}^* \approx 1 - \frac{2}{\kappa}
$$

当条件数 $\kappa$ 很大时，HB法的收敛因子比GD法**小得多**。这意味着HB法在病态问题（低曲率方向）上实现了显著的加速收敛，这就是正文中所说的"**平方根加速**"效应。

> **再次强调**：上述推导针对的是**强凸二次函数**场景下的重球法（HB）。对于一般光滑凸函数，$O(1/T^2)$ 的收敛速率保证来自 Nesterov 加速梯度（NAG），而非 HB。对一般强凸光滑函数，$O(\sqrt{\kappa})$ 的全局加速保证也属于 NAG，而非 HB。详见**附录I**。

### 附录H：证明HB算法几乎必然逃离所有严格鞍点

https://chat.deepseek.com/share/uciwf43fp7yxrkkj41

### 附录I：HB法的由来

https://chat.deepseek.com/share/1c5h8pdkz0twfs48t4


### 附录J：HB理论保证的适用边界——为什么"仅强凸二次"？

本附录系统回答一个关键问题：**HB 的完备加速收敛保证，究竟在什么条件下成立？**

---

#### J.1 核心命题

> **HB（重球法 / Polyak 动量）具有完备、全局、最优加速收敛保证的标准理论结果，基本上只对强凸二次函数成立。**

更精确地说：

- 对**强凸二次函数**，HB 有完整、干净、不依赖额外修正的全局线性收敛保证，且达到与 NAG 同阶的最优速率 $O(\sqrt{\kappa}\log(1/\epsilon))$。
- 对**一般强凸光滑函数**，固定 $\alpha, \beta$ 的经典 HB **没有**全局加速保证，甚至可能不收敛。
- 对**凸但非强凸**或**非凸**函数，HB 更无完备的加速理论。

---

#### J.2 强凸二次函数：完备的加速理论

考虑

$$
\min_x\; f(x)=\frac12 x^T A x - b^T x,\qquad A\succ 0
$$

HB 更新：

$$
x_{k+1}=x_k-\alpha \nabla f(x_k)+\beta (x_k-x_{k-1})
$$

取最优参数

$$
\alpha=\frac{4}{(\sqrt{L}+\sqrt{\mu})^2},\qquad
\beta=\left(\frac{\sqrt{L}-\sqrt{\mu}}{\sqrt{L}+\sqrt{\mu}}\right)^2
$$

其中 $L=\lambda_{\max}(A)$，$\mu=\lambda_{\min}(A)$，则有

$$
\|x_k-x^*\| \le
\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^k \|x_0-x^*\|
$$

其中 $\kappa=L/\mu$。

这是**与 Nesterov 加速梯度法同阶的最优线性收敛速率**，且证明完整、参数最优。附录 G 给出的正是这一情形下的完整推导。

---

#### J.3 离开强凸二次之后：理论不完整

##### J.3.1 一般强凸光滑函数

对一般 $L$-光滑、$\mu$-强凸函数，HB **没有**与二次情形相同的全局加速保证。

原因：

- 动量项 $\beta(x_k-x_{k-1})$ 可能使函数值上升；
- 缺少 Nesterov 方法中"梯度在预测点计算"的结构；
- 已有反例表明：固定 $\alpha,\beta$ 的 HB 在一般强凸函数上可能不收敛，或收敛速度远差于 GD。

因此，**一般强凸光滑函数上，HB 没有完备的加速收敛理论**。Nesterov 加速梯度法才是具有严格全局加速保证的方法。

##### J.3.2 非强凸、凸或非凸情形

更弱：

- 凸但非强凸：HB 无 $O(1/k^2)$ 加速保证；
- 非凸：只有弱收敛结果或需要小步长、特定修正；
- 实际中 HB 常用，但理论主要依赖二次模型或局部二次近似。

---

#### J.4 更精确的结论

所以命题"HB 完备的加速收敛保证，仅对强凸二次函数成立"应修正为：

> **Heavy-Ball 方法具有完备、全局、最优加速收敛保证的标准理论结果，基本上只对强凸二次函数成立。**
>
> 对一般强凸光滑函数，它没有与 NAG 同级的全局加速保证；对非强凸或非二次问题，更没有完备加速理论。

因此：

- 若"完备的加速收敛保证"指：全局、最优、严格可证明的线性加速收敛，那么**是的，仅强凸二次**。
- 若允许局部、修改或额外正则条件，则 HB 可推广到部分非二次情形，但那已不是经典 HB 的完备保证。

---

#### J.5 与 NAG 的理论对比

| 函数类 | GD | HB（经典固定参数） | NAG |
| :--- | :--- | :--- | :--- |
| 强凸二次 | $O(\kappa\log(1/\epsilon))$ | $O(\sqrt{\kappa}\log(1/\epsilon))$ ✅ | $O(\sqrt{\kappa}\log(1/\epsilon))$ ✅ |
| 一般强凸光滑 | $O(\kappa\log(1/\epsilon))$ | 无全局加速保证 ⚠️ | $O(\sqrt{\kappa}\log(1/\epsilon))$ ✅ |
| 凸光滑 | $O(1/T)$ | 无 $O(1/T^2)$ 保证 ⚠️ | $O(1/T^2)$ ✅ |
| 非凸光滑 | 弱收敛 | 弱收敛/经验性 | 弱收敛/经验性 |

---

#### J.6 实践含义

1. **理论视角**：HB 的"加速"在数学上严格成立的范围很窄——强凸二次。离开这个范围，它缺乏 NAG 那样的全局加速保证。

2. **实践视角**：在深度学习中，HB 仍然表现良好，但这主要是**经验性的**，而非严格理论保证的结果。其逃逸鞍点、平滑路径等能力来自惯性机制本身，而非可证明的加速定理。

3. **选择建议**：
   - 如果追求**严格理论保证**的加速：选 NAG；
   - 如果追求**简单、稳定、实践效果好**：HB 仍是合理选择；
   - 在深度学习框架中，`momentum` 参数默认实现的是 HB，而 `nesterov=True` 则切换为 NAG。

### 附录K：稳定流形（Stable Manifold）详解

本附录详细阐述**稳定流形**的概念、数学定义、几何性质，以及它在HB法逃逸鞍点过程中的关键作用。该概念在正文**4.5.3节**中首次提及，并在**第1节实验**中有直接体现。

---

#### K.1 概念引入：什么是稳定流形？

在非线性动力系统和优化理论中，**稳定流形（Stable Manifold）**是鞍点处的一个重要几何结构。对于目标函数 $f(x,y)$ 的鞍点 $x^*$，稳定流形定义为：

$$
W^s(x^*) = \{x \in \mathbb{R}^n \mid \lim_{t \to \infty} \phi_t(x) = x^*\}
$$

其中 $\phi_t$ 是梯度流 $\dot{x} = -\nabla f(x)$ 的演化算子。直观理解：

- **稳定流形**：从这些点出发，沿梯度下降方向**最终收敛到鞍点**的点集
- **不稳定流形**：从这些点出发，沿梯度下降方向**远离鞍点**的点集

在2D问题中，稳定流形和不稳定流形都是**1维流形**，即一条曲线。在鞍点附近，它们是互相垂直的两条线。

---

#### K.2 数学定义：Hessian特征方向分析

对于鞍点 $x^*$，Hessian矩阵 $\nabla^2 f(x^*)$ 的特征值有正有负：

- **正特征值方向** → 函数值是**极小值**（像山谷）→ **稳定流形方向**
- **负特征值方向** → 函数值是**极大值**（像山脊）→ **不稳定流形方向**

在正文第1节的目标函数 $f(x,y) = 0.2(x^2-1)^2 + y^2 - 0.15x$ 中，鞍点 $(-0.1949, 0)$ 的Hessian为：

$$
\nabla^2 f(-0.1949, 0) = 
\begin{bmatrix}
0.8(3(-0.1949)^2-1) & 0 \\
0 & 2
\end{bmatrix}
\approx
\begin{bmatrix}
-0.709 & 0 \\
0 & 2
\end{bmatrix}
$$

特征值分解：

| 特征值 | 特征方向 | 类型 | 对应的流形 |
| :--- | :--- | :--- | :--- |
| $\lambda_1 \approx -0.709$ | x方向（水平） | 负 → 极大值 | **不稳定流形**（鞍点脊线） |
| $\lambda_2 = 2$ | y方向（垂直） | 正 → 极小值 | **稳定流形** |

因此，**稳定流形的方程为 $x = -0.1949$**（过鞍点的垂直线）。

---

#### K.3 几何意义：稳定流形的可视化

在等高线图上，稳定流形是**垂直于鞍点脊线**的一条线：

```
          y
          ^
          |
          |  稳定流形 (x = -0.1949)
          |        |
          |        |
          |        *  ← 鞍点 (-0.1949, 0)
          |        |
          |        |
          |        |
          +--------+--------→ x
        不稳定流形 (y = 0)
```

图中：
- **垂直虚线**（x = -0.1949）：稳定流形
- **水平虚线**（y = 0）：不稳定流形（鞍点脊线）

---

#### K.4 稳定流形在HB法逃逸过程中的作用

在正文4.5.3节中，HB法从起点 $(-0.1949, 1.0)$ 出发：

1. **初始状态**：起点恰好位于稳定流形上（x = -0.1949）
2. **GD法行为**：沿稳定流形方向（y方向）快速下降至鞍点 $(-0.1949, 0)$，然后停滞
3. **HB法行为**：凭借动量惯性，穿越稳定流形，继续向全局最优前进

**为什么GD法会在稳定流形上停滞？**

当GD法从起点 $(-0.1949, 1.0)$ 出发时：

- x方向梯度：$\partial f/\partial x = 0.8(-0.1949)((-0.1949)^2-1) - 0.15 \approx 0$
- y方向梯度：$\partial f/\partial y = 2(1.0) = 2.0$

因此GD法**仅沿y方向下降**，到达鞍点 $(-0.1949, 0)$ 后，梯度在两个方向都趋近于零，更新停滞。

**为什么HB法能穿越稳定流形？**

HB法从稳定流形上的点出发时：

1. **初始阶段**：y方向梯度主导，速度 $v_y$ 快速累积
2. **接近鞍点**：梯度逐渐消失，但速度 $v_y$ 仍然保留（以 $\beta$ 比例衰减）
3. **穿越鞍点**：由于 $x$ 方向存在微小的数值扰动（浮点误差或梯度计算误差），动量可能推动参数在x方向产生偏移
4. **离开稳定流形**：一旦x方向偏移，梯度在x方向变为非零，HB法顺势沿x方向前进，最终到达全局最优

---

#### K.5 稳定流形上的关键点

对于正文第1节的目标函数，稳定流形方程 $x = -0.1949$ 上的几个关键点：

| y 坐标 | 点坐标 | 函数值 f(x,y) | 梯度 (∂f/∂x, ∂f/∂y) |
| :--- | :--- | :--- | :--- |
| -1.0 | (-0.1949, -1.0) | 1.0754 | (0.0, -2.0) |
| 0.0 | (-0.1949, 0.0) | 0.0754 | (0.0, 0.0) ← 鞍点 |
| 0.5 | (-0.1949, 0.5) | 0.3254 | (0.0, 1.0) |
| 1.0 | (-0.1949, 1.0) | 1.0754 | (0.0, 2.0) ← 起点 |

关键观察：

- 在稳定流形上，**x方向梯度恒为零**（$\partial f/\partial x = 0$）
- 在稳定流形上，**y方向梯度不为零**（$\partial f/\partial y = 2y$），仅在鞍点处为零
- 起点 $(-0.1949, 1.0)$ 恰好位于稳定流形上

---

#### K.6 稳定流形与鞍点脊线的对比

| 特征 | 稳定流形 | 鞍点脊线（不稳定流形） |
| :--- | :--- | :--- |
| **方程** | $x = -0.1949$（垂直线） | $y = 0$（水平线） |
| **方向** | 沿y轴方向 | 沿x轴方向 |
| **Hessian特征值** | 正（$\lambda = 2$） | 负（$\lambda \approx -0.709$） |
| **鞍点处的函数值** | 极小值（山谷） | 极大值（山脊） |
| **GD法行为** | 沿该方向收敛到鞍点 | 沿该方向逃离鞍点 |
| **HB法行为** | 有动量时可能穿越 | 有动量时容易穿越 |

---

#### K.7 稳定流形的计算与可视化

在代码中，稳定流形通过以下方式计算并绘制：

```python
def find_stable_manifold():
    """
    计算稳定流形。
    稳定流形是垂直于鞍点脊线的方向，即沿 y 方向。
    在鞍点 (-0.1949, 0) 处，稳定流形是 x = -0.1949 的垂直线。
    """
    saddle_x, saddle_y = saddle_pt
    
    # 生成稳定流形上的点（垂直线 x = saddle_x）
    y_vals = np.linspace(CONFIG["y_min"], CONFIG["y_max"], 200)
    stable_x = [saddle_x] * len(y_vals)
    stable_y = y_vals
    
    return stable_x, stable_y
```

在图中，稳定流形以**灰色点线**绘制（不显眼），以区别于红色的迭代路径。

---

#### K.8 为什么GD法容易困在稳定流形上？

GD法从稳定流形上的点出发时，**x方向梯度为零**，因此更新方向完全由y方向梯度决定。

以起点 $(-0.1949, 1.0)$ 为例：

$$
\nabla f(-0.1949, 1.0) = (0.0, 2.0)
$$

GD法更新：

$$
x_{t+1} = x_t - \alpha \cdot 0 = x_t
$$

$$
y_{t+1} = y_t - \alpha \cdot 2.0
$$

因此，GD法**仅在y方向下降**，x坐标始终保持不变。当到达鞍点 $(-0.1949, 0)$ 时，梯度在两个方向都为零，更新完全停滞。

**HB法之所以不同**，是因为动量项 $v_x$ 可能积累微小的数值噪声或来自浮点误差的扰动，使其在x方向产生偏移。一旦x方向产生微小偏移，梯度在x方向变为非零，HB法顺势沿x方向前进，逃离稳定流形。

---

#### K.9 总结

**稳定流形**是鞍点处的重要几何结构：

1. **定义**：从这些点出发，沿梯度下降方向最终收敛到鞍点的点集
2. **方程**：对于本文目标函数，稳定流形为 $x = -0.1949$（垂直线）
3. **方向**：沿Hessian正特征值方向（y方向）
4. **作用**：GD法容易困在稳定流形上，HB法凭借动量惯性可以穿越
5. **可视化**：在等高线图上以灰色点线绘制，与鞍点脊线（不稳定流形）互相垂直

稳定流形是理解优化算法在鞍点附近行为的关键概念，它解释了为什么GD法容易停滞而HB法能继续前进。